In [1]:
# =========================
# Colab: mount Drive first
# =========================
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

urban_flood_modelling_path = kagglehub.competition_download('urban-flood-modelling')
mtmrs1_urban_flood_bench_re_path = kagglehub.dataset_download('mtmrs1/urban-flood-bench-re')

print('Data source import complete.')


100%|██████████| 146M/146M [00:08<00:00, 17.8MB/s]

Extracting files...


100%|██████████| 5.67G/5.67G [04:36<00:00, 22.0MB/s]

Extracting files...


Data source import complete.


## Physics-Informed GNN: Model2 1D Water-Level + Static Features + Graph Attention + Multi-Step Rollout + Rain Gating

This Colab notebook trains a Model2 1D physics-informed GNN for water-level prediction with static features, graph attention, and an auxiliary multi-step rollout loss.

The model also adds a lightweight rain-conditioned FiLM gate immediately after `input_proj`. The gate is driven by the current rain lag block plus event-level rain mean, and conditions the hidden representation before graph message passing.

Core behavior:
- same target and recursive prediction interface as the static 1D baseline family
- same static feature and extra dynamic feature handling
- same graph attention / mean switch and rollout loss option
- same submission / bundle export flow

Added options:
- `RAIN_GATE_ENABLE`
- `RAIN_GATE_HIDDEN`
- `RAIN_GATE_SCALE`


In [4]:
import os
import gc
import time
import copy
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


warnings.filterwarnings("ignore")


# =========================
# Config (Physics-Informed GNN Graph Attention test setup)
# =========================


def env_int(name: str, default: int) -> int:
    return int(os.environ.get(name, str(default)))


def env_float(name: str, default: float) -> float:
    return float(os.environ.get(name, str(default)))


def env_bool(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}


def parse_str_list(text: str):
    return [c.strip() for c in text.split(",") if c.strip()]


SEED = env_int("SEED", 2026)
DATASET_ROOT = next(
    (Path.home() / ".cache/kagglehub/datasets/mtmrs1/urban-flood-bench-re").glob("versions/*"),
    (Path.home() / ".cache/kagglehub/datasets/mtmrs1/urban-flood-bench-re"),
)
SAMPLE_SUBMISSION_PATH = str(
    next(
        (Path.home() / ".cache/kagglehub/competitions/urban-flood-modelling").rglob("sample_submission.csv"),
        Path("/content/sample_submission.csv"),
    )
)
TARGET = (2, 1)  # fixed: model2 1d
TARGET_COL = "water_level"


def default_y_lags(node_type: int) -> int:
    # Keep lag defaults aligned with the existing global baseline family.
    return 10 if node_type == 1 else 4


def default_rain_lags(model_id: int) -> int:
    # Keep lag defaults aligned with the existing global baseline family.
    return 60 if model_id == 1 else 80


TARGET_MODEL_ID, TARGET_NODE_TYPE = TARGET
N_FOLDS = env_int("N_FOLDS", 5)
RUN_SINGLE_FOLD = env_bool("RUN_SINGLE_FOLD", False)
FOLD_INDEX = env_int("FOLD_INDEX", 0)
EVENT_SPLIT_SEED = env_int("EVENT_SPLIT_SEED", 2026)
START_T = env_int("START_T", 10)
Y_LAGS = env_int("Y_LAGS", default_y_lags(TARGET_NODE_TYPE))
RAIN_LAGS = env_int("RAIN_LAGS", default_rain_lags(TARGET_MODEL_ID))
EXTRA_COLS_REQ = parse_str_list(os.environ.get("EXTRA_COLS", ""))
JOINT_AUX_COLS_REQ = parse_str_list(os.environ.get("JOINT_AUX_COLS", "inlet_flow"))
EXTRA_LAGS = env_int("EXTRA_LAGS", 10)
EXTRA_COLS_ACTIVE = EXTRA_COLS_REQ.copy()
JOINT_AUX_COLS_ACTIVE = []
AUX_LOSS_WEIGHT = env_float("AUX_LOSS_WEIGHT", 1.0)
MAX_TRAIN_EVENTS = env_int("MAX_TRAIN_EVENTS", 0)
MAX_TEST_EVENTS = env_int("MAX_TEST_EVENTS", 0)
MAX_TRAIN_ROWS = env_int("MAX_TRAIN_ROWS", 0)
MAX_VAL_ROWS = env_int("MAX_VAL_ROWS", 0)
EPOCHS = env_int("EPOCHS", 300)
LR = env_float("LR", 3e-4)
WEIGHT_DECAY = env_float("WEIGHT_DECAY", 3e-6)
PATIENCE = env_int("PATIENCE", 25)
MIN_LR = env_float("MIN_LR", 1e-5)
LR_FACTOR = env_float("LR_FACTOR", 0.5)
LR_PATIENCE = env_int("LR_PATIENCE", 5)
GRAD_CLIP_NORM = env_float("GRAD_CLIP_NORM", 1.0)
EPOCH_OOF_EVAL_EVERY = env_int("EPOCH_OOF_EVAL_EVERY", 5)
EPOCH_OOF_MAX_EVENTS = env_int("EPOCH_OOF_MAX_EVENTS", 0)
OOF_LB_EVAL = env_bool("OOF_LB_EVAL", True)
OOF_LB_MAX_EVENTS = env_int("OOF_LB_MAX_EVENTS", 0)
GNN_D_MODEL = env_int("GNN_D_MODEL", 128)
GNN_N_LAYERS = env_int("GNN_N_LAYERS", 15)
GNN_DROPOUT = env_float("GNN_DROPOUT", 0.0)
NODE_EMBED_DIM = env_int("NODE_EMBED_DIM", 96)
GNN_LAYER_TYPE = os.environ.get("GNN_LAYER_TYPE", "attn").strip().lower()
ATTN_DROPOUT = env_float("ATTN_DROPOUT", 0.0)
ATTN_USE_OUT_PROJ = env_bool("ATTN_USE_OUT_PROJ", True)
RAIN_GATE_ENABLE = env_bool("RAIN_GATE_ENABLE", True)
RAIN_GATE_HIDDEN = env_int("RAIN_GATE_HIDDEN", 64)
RAIN_GATE_SCALE = env_float("RAIN_GATE_SCALE", 0.5)
GRAPH_BATCH_SIZE = env_int("GRAPH_BATCH_SIZE", 128)
ROLLOUT_LOSS_ENABLE = env_bool("ROLLOUT_LOSS_ENABLE", True)
ROLLOUT_STEPS = env_int("ROLLOUT_STEPS", 3)
ROLLOUT_WEIGHT = env_float("ROLLOUT_WEIGHT", 0.2)
ROLLOUT_BATCH_SIZE = env_int("ROLLOUT_BATCH_SIZE", 128)
ROLLOUT_MAX_SAMPLES = env_int("ROLLOUT_MAX_SAMPLES", 0)
LAMBDA_SMOOTH = env_float("LAMBDA_SMOOTH", 0.05)
LAMBDA_MASS = env_float("LAMBDA_MASS", 0.02)
LAMBDA_DYN = env_float("LAMBDA_DYN", 0.02)
PRED_BATCH_SIZE = env_int("PRED_BATCH_SIZE", 8192)
LOG_EVERY_EVENTS = env_int("LOG_EVERY_EVENTS", 1)
DEFAULT_OUTPUT_PATH = "/kaggle/working/submission.csv" if Path("/kaggle/working").exists() else "submission.csv"
OUTPUT_PATH = os.environ.get("OUTPUT_PATH", DEFAULT_OUTPUT_PATH)
DEFAULT_PRED_TABLE_PATH = "/kaggle/working/pred_physgnn_m2_1d_5fold_static_attn_rollout_raingate.csv" if Path("/kaggle/working").exists() else "pred_physgnn_m2_1d_5fold_static_attn_rollout_raingate.csv"
SAVE_PRED_TABLE = env_bool("SAVE_PRED_TABLE", True)
PRED_TABLE_PATH = os.environ.get("PRED_TABLE_PATH", DEFAULT_PRED_TABLE_PATH)
STATIC_FEATURES_ENABLE = env_bool("STATIC_FEATURES_ENABLE", True)
STATIC_COLS_REQ = parse_str_list(os.environ.get("STATIC_COLS", "position_x,position_y,depth,invert_elevation,surface_elevation,base_area"))
STATIC_STANDARDIZE = env_bool("STATIC_STANDARDIZE", False)
SAVE_BUNDLE = env_bool("SAVE_BUNDLE", True)
DEFAULT_BUNDLE_ROOT = "/kaggle/working/model_bundles" if Path("/kaggle/working").exists() else "ensemble/artifacts/model_bundles"
BUNDLE_ROOT = Path(os.environ.get("BUNDLE_ROOT", DEFAULT_BUNDLE_ROOT))
BUNDLE_NAME = os.environ.get("BUNDLE_NAME", "physics_informed_gnn_m2_1d_5fold_static_attn_rollout_raingate")
BUNDLE_DIR = Path(os.environ.get("BUNDLE_DIR", str(BUNDLE_ROOT / BUNDLE_NAME)))
SAFE_FEATURE_CLIP = 1e6
SAFE_LEVEL_CLIP = 1e6
SAFE_DELTA_CLIP = 1e4
KEY_COLS = ["model_id", "event_id", "node_type", "node_id", "timestep"]
STD_DEV = {
    (1, 1): 16.877747,
    (1, 2): 14.378797,
    (2, 1): 3.191784,
    (2, 2): 2.727131,
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = env_bool("USE_AMP", True) and DEVICE == "cuda"
CPU_COUNT = max(1, int(os.cpu_count() or 1))
NUM_WORKERS = env_int("NUM_WORKERS", 2 if CPU_COUNT >= 4 else 0)
if N_FOLDS < 2:
    raise RuntimeError(f"N_FOLDS must be >=2, got {N_FOLDS}")
if START_T < 1:
    raise RuntimeError(f"START_T must be >=1, got {START_T}")
if Y_LAGS < 1:
    raise RuntimeError(f"Y_LAGS must be >=1, got {Y_LAGS}")
if RAIN_LAGS < 0:
    raise RuntimeError(f"RAIN_LAGS must be >=0, got {RAIN_LAGS}")
if START_T < Y_LAGS:
    raise RuntimeError(
        f"START_T must be >= Y_LAGS for autoregressive safety: START_T={START_T}, Y_LAGS={Y_LAGS}"
    )
if GNN_D_MODEL < 16:
    raise RuntimeError(f"GNN_D_MODEL too small: {GNN_D_MODEL}")
if GNN_LAYER_TYPE not in {"mean", "attn"}:
    raise RuntimeError(f"GNN_LAYER_TYPE must be one of {{'mean', 'attn'}}, got {GNN_LAYER_TYPE}")
if GRAPH_BATCH_SIZE < 1:
    raise RuntimeError(f"GRAPH_BATCH_SIZE must be >=1, got {GRAPH_BATCH_SIZE}")
if ROLLOUT_STEPS < 1:
    raise RuntimeError(f"ROLLOUT_STEPS must be >=1, got {ROLLOUT_STEPS}")
if ROLLOUT_BATCH_SIZE < 1:
    raise RuntimeError(f"ROLLOUT_BATCH_SIZE must be >=1, got {ROLLOUT_BATCH_SIZE}")

# =========================
# Utilities
# =========================


def log(msg: str) -> None:
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_table_auto(df: pd.DataFrame, stem_path: Path) -> Path:
    stem_path.parent.mkdir(parents=True, exist_ok=True)
    parquet_path = stem_path.with_suffix(".parquet")
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        csv_path = stem_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        log(f"parquet save failed ({parquet_path.name}); fallback csv ({csv_path.name}) reason={exc}")
        return csv_path


def resolve_models_root(dataset_root: Path) -> Path:
    cands = []
    seen = set()
    def add(p):
        if p is None:
            return
        p = Path(p)
        key = str(p)
        if key and key not in seen:
            cands.append(p)
            seen.add(key)
    def find_model_root_by_walk(root: Path, max_depth: int = 8):
        if not root.exists():
            return None
        root = root.resolve()
        base_depth = len(root.parts)
        for dirpath, dirnames, _ in os.walk(root):
            cur = Path(dirpath)
            depth = len(cur.parts) - base_depth
            if depth > max_depth:
                dirnames[:] = []
                continue
            lower_names = {d.lower() for d in dirnames}
            if "model_1" in lower_names or "model1" in lower_names:
                return cur
            if cur.name.lower() in {"model_1", "model1"}:
                return cur.parent
        return None
    add(dataset_root)
    add(dataset_root / "Models")
    for env_name in (
        "MODELS_ROOT",
        "DATASET_ROOT",
        "URBAN_FLOOD_ROOT",
        "UFB_DATASET_ROOT",
        "KAGGLEHUB_CACHE_DIR",
    ):
        raw = os.environ.get(env_name, "").strip()
        if raw:
            p = Path(raw)
            add(p)
            add(p / "Models")
    cwd = Path.cwd()
    add(cwd)
    add(cwd / "Models")
    add(cwd / "input")
    add(cwd / "input" / "Models")
    scan_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/input/competitions"),
        Path("/content"),
        Path("/content/drive/MyDrive"),
        Path("/workspace"),
        Path("/mnt/data"),
        Path("/root/.cache/kagglehub"),
        Path("/root/.cache"),
        cwd,
    ]
    for r in scan_roots:
        if r.exists():
            add(r)
            add(r / "Models")
    for c in cands:
        for sub in ("", "Models", "models"):
            base = c if sub == "" else c / sub
            if (base / "Model_1").exists() or (base / "model_1").exists() or (base / "Model1").exists() or (base / "model1").exists():
                return base
    for root in scan_roots:
        if not root.exists():
            continue
        patterns = (
            "*/Model_1",
            "*/*/Model_1",
            "*/*/*/Model_1",
            "*/*/*/*/Model_1",
            "*/Models/Model_1",
            "*/*/Models/Model_1",
            "*/*/*/Models/Model_1",
            "*/*/*/*/Models/Model_1",
        )
        for pat in patterns:
            for p in sorted(root.glob(pat)):
                return p.parent
    for root in scan_roots:
        found = find_model_root_by_walk(root, max_depth=8)
        if found is not None:
            return found
    checked = ", ".join(str(p) for p in cands[:18])
    raise RuntimeError(
        "Models root not found. Set DATASET_ROOT or MODELS_ROOT. "
        "For kagglehub on Colab, DATASET_ROOT often looks like "
        "'/root/.cache/kagglehub/datasets/.../versions/...'. "
        f"checked={checked}"
    )


def resolve_sample_submission(explicit: str, models_root: Path = None) -> Path:
    cands = []
    seen = set()
    def add(p):
        if p is None:
            return
        p = Path(p)
        key = str(p)
        if key and key not in seen:
            cands.append(p)
            seen.add(key)
    if explicit:
        add(explicit)
    env_ss = os.environ.get("SAMPLE_SUBMISSION_PATH", "").strip()
    if env_ss:
        add(env_ss)
    if models_root is not None:
        mr = Path(models_root)
        add(mr / "sample_submission.csv")
        add(mr.parent / "sample_submission.csv")
        add(mr.parent.parent / "sample_submission.csv")
    add("/kaggle/input/urban-flood-modelling/sample_submission.csv")
    add("/kaggle/input/urban-flood-modeling/sample_submission.csv")
    add("/kaggle/input/competitions/urban-flood-modelling/sample_submission.csv")
    add("/kaggle/input/competitions/urban-flood-modeling/sample_submission.csv")
    add("/kaggle/input/flood-comp-dataset/sample_submission.csv")
    scan_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/input/competitions"),
        Path("/content"),
        Path("/content/drive/MyDrive"),
        Path("/workspace"),
        Path("/mnt/data"),
        Path("/root/.cache/kagglehub"),
        Path("/root/.cache"),
        Path.cwd(),
    ]
    for root in scan_roots:
        if not root.exists():
            continue
        for pat in (
            "*/sample_submission.csv",
            "*/*/sample_submission.csv",
            "*/*/*/sample_submission.csv",
            "*/*/*/*/sample_submission.csv",
        ):
            for p in sorted(root.glob(pat)):
                add(p)
    # deep fallback for kagglehub-style cache trees
    for root in scan_roots:
        if not root.exists():
            continue
        base_depth = len(root.resolve().parts)
        for dirpath, _, filenames in os.walk(root):
            cur = Path(dirpath)
            if len(cur.parts) - base_depth > 8:
                continue
            if "sample_submission.csv" in filenames:
                add(cur / "sample_submission.csv")
    existing = [p for p in cands if p.exists()]
    if existing:
        preferred = [p for p in existing if "urban-flood" in str(p).lower()]
        return preferred[0] if preferred else existing[0]
    checked = ", ".join(str(p) for p in cands[:20])
    raise RuntimeError(
        "sample submission not found. Set SAMPLE_SUBMISSION_PATH. "
        f"checked={checked}"
    )


def model_dir(models_root: Path, model_id: int) -> Path:
    p = models_root / f"Model_{model_id}"
    if not (p / "train").exists() or not (p / "test").exists():
        raise RuntimeError(f"invalid model dir: {p}")
    return p


def node_static_name(node_type: int) -> str:
    return "1d_nodes_static.csv" if node_type == 1 else "2d_nodes_static.csv"


def node_dynamic_name(node_type: int) -> str:
    return "1d_nodes_dynamic_all.csv" if node_type == 1 else "2d_nodes_dynamic_all.csv"


def list_event_ids(mdir: Path, split: str):
    out = []
    root = mdir / split
    for d in root.iterdir():
        if d.is_dir() and d.name.startswith("event_"):
            try:
                out.append(int(d.name.split("_")[-1]))
            except ValueError:
                pass
    return sorted(out)


def load_node_ids(mdir: Path, node_type: int):
    df = pd.read_csv(mdir / "train" / node_static_name(node_type), usecols=["node_idx"])
    return np.sort(df["node_idx"].astype(np.int32).unique())


def load_node_area_weights(mdir: Path, node_type: int, node_ids: np.ndarray):
    static_path = mdir / "train" / node_static_name(node_type)
    cols = pd.read_csv(static_path, nrows=0).columns.tolist()
    area_col = None
    for c in ("base_area", "area", "surface_area"):
        if c in cols:
            area_col = c
            break
    if area_col is None:
        return np.ones((len(node_ids),), dtype=np.float32)
    df = pd.read_csv(static_path, usecols=["node_idx", area_col])
    s = df.set_index("node_idx")[area_col].reindex(node_ids)
    arr = s.to_numpy(dtype=np.float32)
    np.nan_to_num(arr, copy=False, nan=1.0, posinf=1.0, neginf=1.0)
    arr = np.clip(arr, 1e-6, 1e6).astype(np.float32, copy=False)
    return arr


def load_static_node_features(
    mdir: Path,
    node_type: int,
    node_ids: np.ndarray,
    requested_cols,
    standardize: bool,
):
    static_path = mdir / "train" / node_static_name(node_type)
    full = pd.read_csv(static_path)
    if "node_idx" not in full.columns:
        raise RuntimeError(f"node_idx missing in static file: {static_path}")
    work = full.set_index("node_idx").reindex(node_ids)
    numeric_cols = []
    for col in work.columns.tolist():
        if col == "node_idx":
            continue
        if pd.api.types.is_numeric_dtype(work[col]):
            numeric_cols.append(col)
    if requested_cols:
        active_cols = [c for c in requested_cols if c in numeric_cols]
        missing = [c for c in requested_cols if c not in numeric_cols]
        if missing:
            log(f"static columns missing and skipped: {missing}")
    else:
        active_cols = numeric_cols
    if not STATIC_FEATURES_ENABLE or not active_cols:
        return np.empty((len(node_ids), 0), dtype=np.float32), active_cols
    feat = work[active_cols].to_numpy(dtype=np.float32, copy=False)
    col_mean = np.nanmean(feat, axis=0).astype(np.float32, copy=False)
    col_mean = np.where(np.isfinite(col_mean), col_mean, 0.0).astype(np.float32, copy=False)
    mask = ~np.isfinite(feat)
    if mask.any():
        feat = np.where(mask, col_mean[None, :], feat).astype(np.float32, copy=False)
    if standardize and feat.shape[1] > 0:
        mu = feat.mean(axis=0, dtype=np.float64).astype(np.float32)
        sigma = feat.std(axis=0, dtype=np.float64).astype(np.float32)
        sigma = np.where(sigma < 1e-6, 1.0, sigma).astype(np.float32)
        feat = ((feat - mu[None, :]) / sigma[None, :]).astype(np.float32, copy=False)
    np.nan_to_num(feat, copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
    np.clip(feat, -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=feat)
    return feat.astype(np.float32, copy=False), active_cols


def load_1d_edge_index(mdir: Path, node_ids: np.ndarray):
    cands = [
        mdir / "train" / "1d_edge_index.csv",
        mdir / "test" / "1d_edge_index.csv",
        mdir / "train" / "1d_edges_index.csv",
        mdir / "test" / "1d_edges_index.csv",
    ]
    edge_path = None
    for p in cands:
        if p.exists():
            edge_path = p
            break
    if edge_path is None:
        raise RuntimeError("1d_edge_index.csv not found in train/test")
    df = pd.read_csv(edge_path)
    cols = {c.lower(): c for c in df.columns}
    from_col = None
    to_col = None
    for key in ("from_node", "src_node", "source_node", "u", "from"):
        if key in cols:
            from_col = cols[key]
            break
    for key in ("to_node", "dst_node", "target_node", "v", "to"):
        if key in cols:
            to_col = cols[key]
            break
    if from_col is None or to_col is None:
        raise RuntimeError(f"edge index columns not found in {edge_path}; cols={list(df.columns)}")
    id2pos = {int(nid): i for i, nid in enumerate(node_ids.tolist())}
    src = df[from_col].astype(np.int32).to_numpy()
    dst = df[to_col].astype(np.int32).to_numpy()
    edges = []
    for s, d in zip(src.tolist(), dst.tolist()):
        if s in id2pos and d in id2pos:
            u = id2pos[s]
            v = id2pos[d]
            edges.append((u, v))
            edges.append((v, u))  # undirected expand
    if not edges:
        raise RuntimeError("no valid 1D edges after mapping edge_index to node_ids")
    edges = np.asarray(edges, dtype=np.int64)
    edge_index = np.stack([edges[:, 0], edges[:, 1]], axis=0)
    return edge_index


def detect_rain_col(mdir: Path, split: str, event_id: int):
    evdir = mdir / split / f"event_{event_id}" / "2d_nodes_dynamic_all.csv"
    cols = pd.read_csv(evdir, nrows=1).columns.tolist()
    for c in ("rainfall", "rainfall_depth", "rainfall_intensity"):
        if c in cols:
            return c
    raise RuntimeError("rainfall column not found")


def resolve_extra_cols(mdir: Path, split: str, event_id: int, node_type: int, requested_cols):
    if not requested_cols:
        return []
    dyn_path = mdir / split / f"event_{event_id}" / node_dynamic_name(node_type)
    cols = pd.read_csv(dyn_path, nrows=0).columns.tolist()
    active = [c for c in requested_cols if c in cols and c != TARGET_COL]
    missing = [c for c in requested_cols if c not in cols]
    if missing:
        log(f"extra columns missing and skipped: {missing}")
    return active


def build_rain_lags(rain: np.ndarray, rain_lags: int) -> np.ndarray:
    n = len(rain)
    out = np.zeros((n, rain_lags + 1), dtype=np.float32)
    if n == 0:
        return out
    out[:, 0] = rain
    for k in range(1, rain_lags + 1):
        if k >= n:
            break
        out[k:, k] = rain[: n - k]
    return out


def load_event(mdir: Path, split: str, event_id: int, node_type: int, node_ids: np.ndarray, target_col: str, rain_col: str):
    evdir = mdir / split / f"event_{event_id}"
    dyn_path = evdir / node_dynamic_name(node_type)
    cols = pd.read_csv(dyn_path, nrows=0).columns.tolist()
    extra_cols_present = [c for c in EXTRA_COLS_ACTIVE if c in cols and c != target_col]
    usecols = ["timestep", "node_idx", target_col] + extra_cols_present
    df = pd.read_csv(dyn_path, usecols=usecols)
    p = (
        df.pivot(index="timestep", columns="node_idx", values=target_col)
        .sort_index()
        .reindex(columns=node_ids)
    )
    r = pd.read_csv(evdir / "2d_nodes_dynamic_all.csv", usecols=["timestep", rain_col])
    rain = r.drop_duplicates("timestep").sort_values("timestep")[rain_col].to_numpy(dtype=np.float32)
    t = p.index.to_numpy(dtype=np.int32)
    y = p.to_numpy(dtype=np.float32)
    extra = {}
    for col in EXTRA_COLS_ACTIVE:
        if col in extra_cols_present:
            px = (
                df.pivot(index="timestep", columns="node_idx", values=col)
                .sort_index()
                .reindex(columns=node_ids)
                .reindex(index=p.index)
            )
            extra[col] = px.to_numpy(dtype=np.float32)
        else:
            extra[col] = np.full_like(y, np.nan, dtype=np.float32)
    n = min(len(t), len(rain), y.shape[0])
    return {
        "event_id": int(event_id),
        "t": t[:n],
        "y": y[:n],
        "rain": rain[:n],
        "extra": {k: v[:n] for k, v in extra.items()},
    }


def split_events_kfold(events, n_folds: int, seed: int):
    arr = np.asarray(sorted(events), dtype=np.int32)
    rng = np.random.RandomState(seed)
    rng.shuffle(arr)
    folds = np.array_split(arr, n_folds)
    return [sorted(x.tolist()) for x in folds]


def fold_train_events(folds, holdout_idx: int):
    out = []
    for i, f in enumerate(folds):
        if i == holdout_idx:
            continue
        out.extend(f)
    return sorted(out)


def compute_node_means_from_events(events):
    n_nodes = events[0]["y"].shape[1]
    sums = np.zeros((n_nodes,), dtype=np.float64)
    cnt = 0
    for ev in events:
        arr = ev["y"]
        sums += np.nan_to_num(arr, nan=0.0).sum(axis=0)
        cnt += arr.shape[0]
    return (sums / max(cnt, 1)).astype(np.float32)


def compute_extra_means_from_events(events, extra_cols):
    out = {}
    for col in extra_cols:
        n_nodes = events[0]["y"].shape[1]
        sums = np.zeros((n_nodes,), dtype=np.float64)
        cnt = 0
        for ev in events:
            arr = ev["extra"][col]
            sums += np.nan_to_num(arr, nan=0.0).sum(axis=0)
            cnt += arr.shape[0]
        out[col] = (sums / max(cnt, 1)).astype(np.float32)
    return out


def fill_missing(events, node_means, extra_means=None):
    for ev in events:
        arr = ev["y"]
        mask = ~np.isfinite(arr)
        if mask.any():
            arr = np.where(mask, node_means[np.newaxis, :], arr).astype(np.float32, copy=False)
            ev["y"] = arr
        if extra_means:
            for col, means in extra_means.items():
                ex = ev["extra"][col]
                ex_mask = ~np.isfinite(ex)
                if ex_mask.any():
                    ex = np.where(ex_mask, means[np.newaxis, :], ex).astype(np.float32, copy=False)
                    ev["extra"][col] = ex


def choose_time_indices(events, start_t: int, max_rows: int, n_nodes: int, seed: int):
    rows_possible = 0
    for ev in events:
        rows_possible += max(0, len(ev["t"]) - start_t) * n_nodes
    if max_rows <= 0 or rows_possible <= max_rows:
        return [None] * len(events), rows_possible, rows_possible
    rng = np.random.RandomState(seed)
    picks = []
    rows_kept = 0
    for ev in events:
        n = len(ev["t"])
        if n <= start_t:
            picks.append(np.array([], dtype=np.int32))
            continue
        rows_ev = (n - start_t) * n_nodes
        keep_t = max(1, int(round((rows_ev / rows_possible) * max_rows)))
        keep_t = min(keep_t, n - start_t)
        idx = rng.choice(n - start_t, size=keep_t, replace=False).astype(np.int32)
        picks.append(idx)
        rows_kept += keep_t * n_nodes
    return picks, rows_possible, rows_kept


def build_samples(
    events,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
    static_feat: np.ndarray | None,
    max_rows: int,
    seed: int,
):
    n_nodes = events[0]["y"].shape[1]
    node_pos = np.arange(n_nodes, dtype=np.float32) / max(1.0, float(n_nodes - 1))
    picks, rows_possible, rows_kept = choose_time_indices(events, start_t, max_rows, n_nodes, seed)
    extra_dim = len(extra_cols) * max(0, extra_lags)
    static_dim = 0 if static_feat is None else int(static_feat.shape[1])
    feat_dim = 1 + static_dim + y_lags + extra_dim + (rain_lags + 1) + 1
    x = np.empty((rows_kept, feat_dim), dtype=np.float32)
    y = np.empty((rows_kept,), dtype=np.float32)
    n_aux = len(joint_aux_cols)
    y_aux = np.empty((rows_kept, n_aux), dtype=np.float32) if n_aux > 0 else np.empty((rows_kept, 0), dtype=np.float32)
    ptr = 0
    total = len(events)
    for j, ev in enumerate(events, 1):
        arr = ev["y"]
        rain = ev["rain"]
        extra = ev["extra"]
        n = min(arr.shape[0], len(rain))
        if n <= start_t:
            continue
        arr = arr[:n]
        rain = rain[:n]
        extra = {k: v[:n] for k, v in extra.items()}
        rain_hist = build_rain_lags(rain, rain_lags)
        rain_mean = float(np.mean(rain))
        rel = picks[j - 1]
        if rel is None:
            rel = np.arange(n - start_t, dtype=np.int32)
        if rel.size == 0:
            continue
        for r in rel.tolist():
            tt = start_t + int(r)
            step_y = np.stack([arr[tt - 1 - k, :] for k in range(y_lags)], axis=1).astype(
                np.float32, copy=False
            )
            step_extra = []
            if extra_lags > 0:
                for col in extra_cols:
                    ex = extra[col]
                    ex_lag = np.stack(
                        [ex[tt - 1 - k, :] for k in range(extra_lags)],
                        axis=1,
                    ).astype(np.float32, copy=False)
                    step_extra.append(ex_lag)
            step_r = np.tile(rain_hist[tt, : rain_lags + 1][None, :], (n_nodes, 1)).astype(
                np.float32, copy=False
            )
            step_mean = np.full((n_nodes, 1), rain_mean, dtype=np.float32)
            feat_parts = [node_pos[:, None]]
            if static_feat is not None and static_feat.shape[1] > 0:
                feat_parts.append(static_feat)
            feat_parts.append(step_y)
            if step_extra:
                feat_parts.append(np.concatenate(step_extra, axis=1))
            feat_parts.extend([step_r, step_mean])
            feat = np.concatenate(feat_parts, axis=1)
            target = (arr[tt, :] - arr[tt - 1, :]).astype(np.float32, copy=False)
            if n_aux > 0:
                target_aux = np.stack(
                    [(extra[col][tt, :] - extra[col][tt - 1, :]).astype(np.float32, copy=False) for col in joint_aux_cols],
                    axis=1,
                ).astype(np.float32, copy=False)
            x[ptr : ptr + n_nodes, :] = feat
            y[ptr : ptr + n_nodes] = target
            if n_aux > 0:
                y_aux[ptr : ptr + n_nodes, :] = target_aux
            ptr += n_nodes
        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
            log(f"sample build progress: {j}/{total} events")
    x = x[:ptr]
    y = y[:ptr]
    y_aux = y_aux[:ptr]
    np.nan_to_num(x, copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
    np.clip(x, -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=x)
    np.nan_to_num(y, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    if n_aux > 0:
        np.nan_to_num(y_aux, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return x, y, y_aux, rows_possible, ptr


def choose_rollout_sequence_indices(events, start_t: int, rollout_steps: int, max_samples: int, seed: int):
    counts = []
    total = 0
    for ev in events:
        n = min(ev["y"].shape[0], len(ev["rain"]))
        c = max(0, n - start_t - rollout_steps + 1)
        counts.append(c)
        total += c
    if max_samples <= 0 or total <= max_samples:
        picks = [None if c > 0 else np.array([], dtype=np.int32) for c in counts]
        return picks, total, total
    rng = np.random.RandomState(seed)
    picks = []
    kept = 0
    for c in counts:
        if c <= 0:
            picks.append(np.array([], dtype=np.int32))
            continue
        keep = max(1, int(round((c / max(total, 1)) * max_samples)))
        keep = min(keep, c)
        idx = rng.choice(c, size=keep, replace=False).astype(np.int32)
        picks.append(np.sort(idx))
        kept += keep
    return picks, total, kept


def build_rollout_samples(
    events,
    start_t: int,
    rollout_steps: int,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
    max_samples: int,
    seed: int,
):
    if not events:
        return None, 0, 0
    n_extra = len(extra_cols)
    picks, samples_possible, _ = choose_rollout_sequence_indices(
        events, start_t=start_t, rollout_steps=rollout_steps, max_samples=max_samples, seed=seed
    )
    y_hist_list = []
    y_future_list = []
    rain_steps_list = []
    rain_mean_list = []
    extra_hist_list = []
    extra_future_list = []
    total = len(events)
    for j, ev in enumerate(events, 1):
        arr = ev["y"]
        rain = ev["rain"]
        extra = ev["extra"]
        n = min(arr.shape[0], len(rain))
        if n < start_t + rollout_steps:
            continue
        arr = arr[:n]
        rain = rain[:n]
        extra = {k: v[:n] for k, v in extra.items()}
        rain_hist = build_rain_lags(rain, rain_lags)
        rel = picks[j - 1]
        if rel is None:
            rel = np.arange(n - start_t - rollout_steps + 1, dtype=np.int32)
        if rel.size == 0:
            continue
        rain_mean = float(np.mean(rain))
        for r in rel.tolist():
            tt = start_t + int(r)
            y_hist = np.stack([arr[tt - 1 - k, :] for k in range(y_lags)], axis=0).astype(np.float32, copy=False)
            y_future = arr[tt : tt + rollout_steps, :].astype(np.float32, copy=False)
            rain_steps = rain_hist[tt : tt + rollout_steps, : rain_lags + 1].astype(np.float32, copy=False)
            y_hist_list.append(y_hist)
            y_future_list.append(y_future)
            rain_steps_list.append(rain_steps)
            rain_mean_list.append(rain_mean)
            if n_extra > 0 and extra_lags > 0:
                ex_hist_cols = []
                ex_future_cols = []
                for col in extra_cols:
                    ex = extra[col]
                    ex_hist_cols.append(np.stack([ex[tt - 1 - k, :] for k in range(extra_lags)], axis=0).astype(np.float32, copy=False))
                    ex_future_cols.append(ex[tt : tt + rollout_steps, :].astype(np.float32, copy=False))
                extra_hist_list.append(np.stack(ex_hist_cols, axis=0))
                extra_future_list.append(np.stack(ex_future_cols, axis=0))
        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
            log(f"rollout sample build progress: {j}/{total} events")
    if not y_hist_list:
        return None, samples_possible, 0
    data = {
        "y_hist": np.stack(y_hist_list, axis=0).astype(np.float32, copy=False),
        "y_future": np.stack(y_future_list, axis=0).astype(np.float32, copy=False),
        "rain_steps": np.stack(rain_steps_list, axis=0).astype(np.float32, copy=False),
        "rain_mean": np.asarray(rain_mean_list, dtype=np.float32),
        "count": int(len(y_hist_list)),
    }
    if n_extra > 0 and extra_lags > 0 and extra_hist_list:
        data["extra_hist"] = np.stack(extra_hist_list, axis=0).astype(np.float32, copy=False)
        data["extra_future"] = np.stack(extra_future_list, axis=0).astype(np.float32, copy=False)
    else:
        data["extra_hist"] = None
        data["extra_future"] = None
    return data, samples_possible, int(len(y_hist_list))


def fit_scaler(x: np.ndarray):
    mean = x.mean(axis=0, dtype=np.float64).astype(np.float32)
    std = x.std(axis=0, dtype=np.float64).astype(np.float32)
    std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
    return mean, std


def apply_scaler(x: np.ndarray, mean: np.ndarray, std: np.ndarray):
    return ((x - mean[None, :]) / std[None, :]).astype(np.float32, copy=False)


def apply_scaler_torch(x: torch.Tensor, mean: torch.Tensor, std: torch.Tensor):
    return (x - mean.view(1, 1, -1)) / std.view(1, 1, -1)


# =========================
# Physics-Informed GNN (self-contained)
# =========================


def flat_to_graph_samples(x_flat: np.ndarray, y_flat: np.ndarray, n_nodes: int, tag: str):
    if n_nodes <= 0:
        raise RuntimeError("n_nodes must be > 0")
    if len(y_flat) == 0:
        return np.empty((0, n_nodes, x_flat.shape[1]), dtype=np.float32), np.empty((0, n_nodes), dtype=np.float32)
    if len(y_flat) % n_nodes != 0:
        raise RuntimeError(
            f"{tag}: rows must be divisible by n_nodes: rows={len(y_flat)} n_nodes={n_nodes}"
        )
    n_graph = len(y_flat) // n_nodes
    xg = x_flat.reshape(n_graph, n_nodes, x_flat.shape[1]).astype(np.float32, copy=False)
    yg = y_flat.reshape(n_graph, n_nodes).astype(np.float32, copy=False)
    return xg, yg


def flat_to_graph_aux_samples(y_aux_flat: np.ndarray, n_nodes: int, n_aux: int, tag: str):
    if n_aux <= 0:
        return None
    if y_aux_flat is None or y_aux_flat.size == 0:
        return np.empty((0, n_nodes, n_aux), dtype=np.float32)
    if y_aux_flat.ndim != 2 or y_aux_flat.shape[1] != n_aux:
        raise RuntimeError(f"{tag}: aux target shape mismatch: got {y_aux_flat.shape}, n_aux={n_aux}")
    if y_aux_flat.shape[0] % n_nodes != 0:
        raise RuntimeError(
            f"{tag}: aux rows must be divisible by n_nodes: rows={y_aux_flat.shape[0]} n_nodes={n_nodes}"
        )
    n_graph = y_aux_flat.shape[0] // n_nodes
    return y_aux_flat.reshape(n_graph, n_nodes, n_aux).astype(np.float32, copy=False)


class PhysGraphLayer(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.self_lin = nn.Linear(d_model, d_model)
        self.msg_lin = nn.Linear(d_model, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor):
        src, dst = edge_index
        msg = self.msg_lin(h[src])
        if h.dtype != msg.dtype:
            h = h.to(msg.dtype)
        agg = torch.zeros((h.shape[0], h.shape[1]), device=h.device, dtype=msg.dtype)
        agg.index_add_(0, dst, msg)
        deg = torch.zeros((h.shape[0], 1), device=h.device, dtype=msg.dtype)
        deg.index_add_(0, dst, torch.ones((dst.shape[0], 1), device=h.device, dtype=msg.dtype))
        agg = agg / deg.clamp_min(1.0)
        u = torch.relu(self.self_lin(h) + agg)
        h = self.norm1(h + u)
        h = self.norm2(h + self.ffn(h))
        return h


def edge_softmax_by_dst(scores: torch.Tensor, dst: torch.Tensor, num_nodes: int) -> torch.Tensor:
    if scores.ndim != 1:
        raise RuntimeError(f"edge softmax expects 1D scores, got shape={tuple(scores.shape)}")
    if dst.ndim != 1:
        raise RuntimeError(f"edge softmax expects 1D dst index, got shape={tuple(dst.shape)}")
    if scores.shape[0] != dst.shape[0]:
        raise RuntimeError(f"edge softmax size mismatch: scores={scores.shape[0]} dst={dst.shape[0]}")
    if scores.numel() == 0:
        return scores
    if hasattr(torch.Tensor, "scatter_reduce_"):
        max_per_dst = torch.full((num_nodes,), float("-inf"), device=scores.device, dtype=scores.dtype)
        max_per_dst.scatter_reduce_(0, dst, scores, reduce="amax", include_self=True)
        stable_scores = scores - max_per_dst[dst]
        exp_scores = torch.exp(stable_scores)
        denom = torch.zeros((num_nodes,), device=scores.device, dtype=exp_scores.dtype)
        denom.index_add_(0, dst, exp_scores)
        return exp_scores / denom[dst].clamp_min(1e-12)
    alpha = torch.zeros_like(scores)
    for node_idx in dst.unique(sorted=False).tolist():
        mask = dst == int(node_idx)
        alpha[mask] = torch.softmax(scores[mask], dim=0)
    return alpha


class PhysGraphAttentionLayer(nn.Module):
    def __init__(self, d_model: int, dropout: float, attn_dropout: float = 0.0, use_out_proj: bool = True):
        super().__init__()
        self.self_lin = nn.Linear(d_model, d_model)
        self.q_lin = nn.Linear(d_model, d_model)
        self.k_lin = nn.Linear(d_model, d_model)
        self.v_lin = nn.Linear(d_model, d_model)
        self.out_lin = nn.Linear(d_model, d_model) if use_out_proj else nn.Identity()
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.scale = float(d_model) ** -0.5
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, h: torch.Tensor, edge_index: torch.Tensor):
        src, dst = edge_index
        q = self.q_lin(h)
        k = self.k_lin(h)
        v = self.v_lin(h)
        score = torch.sum(q[dst] * k[src], dim=-1) * self.scale
        alpha = edge_softmax_by_dst(score, dst, h.shape[0])
        alpha = self.attn_dropout(alpha)
        msg = v[src] * alpha.unsqueeze(-1)
        if h.dtype != msg.dtype:
            h = h.to(msg.dtype)
        agg = torch.zeros((h.shape[0], h.shape[1]), device=h.device, dtype=msg.dtype)
        agg.index_add_(0, dst, msg)
        agg = self.out_lin(agg)
        u = torch.relu(self.self_lin(h) + agg)
        h = self.norm1(h + u)
        h = self.norm2(h + self.ffn(h))
        return h


class RainFiLMGate(nn.Module):
    def __init__(self, rain_dim: int, d_model: int, hidden_dim: int, scale: float = 0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rain_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model * 2),
        )
        self.scale = float(scale)
    def forward(self, h_graph: torch.Tensor, rain_ctx: torch.Tensor) -> torch.Tensor:
        if rain_ctx.ndim != 2:
            raise RuntimeError(f"rain_ctx must be 2D, got shape={tuple(rain_ctx.shape)}")
        params = self.net(rain_ctx.to(dtype=h_graph.dtype))
        gamma, beta = params.chunk(2, dim=-1)
        gamma = torch.tanh(gamma) * self.scale
        beta = torch.tanh(beta) * self.scale
        return h_graph * (1.0 + gamma[:, None, :]) + beta[:, None, :]


class PhysicsInformedGNN(nn.Module):
    def __init__(
        self,
        n_features: int,
        n_nodes: int,
        edge_index_np: np.ndarray,
        d_model: int,
        n_layers: int,
        dropout: float,
        node_embed_dim: int,
        n_aux_targets: int = 0,
        layer_type: str = "attn",
        attn_dropout: float = 0.0,
        attn_use_out_proj: bool = True,
        y_lags: int = 10,
        rain_lags: int = 80,
        n_extra_cols: int = 0,
        extra_lags: int = 0,
        static_dim: int = 0,
        rain_gate_enable: bool = True,
        rain_gate_hidden: int = 64,
        rain_gate_scale: float = 0.5,
    ):
        super().__init__()
        self.n_nodes = int(n_nodes)
        self.node_embed_dim = max(0, int(node_embed_dim))
        self.node_emb = nn.Embedding(self.n_nodes, self.node_embed_dim) if self.node_embed_dim > 0 else None
        in_dim = n_features + self.node_embed_dim
        self.input_proj = nn.Linear(in_dim, d_model)
        layer_type = str(layer_type).strip().lower()
        if layer_type == "attn":
            self.layers = nn.ModuleList([
                PhysGraphAttentionLayer(d_model, dropout, attn_dropout=attn_dropout, use_out_proj=attn_use_out_proj)
                for _ in range(max(1, n_layers))
            ])
        elif layer_type == "mean":
            self.layers = nn.ModuleList([PhysGraphLayer(d_model, dropout) for _ in range(max(1, n_layers))])
        else:
            raise RuntimeError(f"unsupported layer_type: {layer_type}")
        self.layer_type = layer_type
        self.n_aux_targets = max(0, int(n_aux_targets))
        self.out_dim = 1 + self.n_aux_targets
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, self.out_dim))
        edge_index = torch.from_numpy(edge_index_np.astype(np.int64, copy=False))
        self.register_buffer("edge_index_base", edge_index, persistent=False)
        self.register_buffer("node_index_base", torch.arange(self.n_nodes, dtype=torch.long), persistent=False)
        self._edge_cache = {}
        self.beta_rain = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))
        extra_dim = int(n_extra_cols) * max(0, int(extra_lags))
        self.rain_start = 1 + int(static_dim) + int(y_lags) + int(extra_dim)
        self.rain_ctx_dim = int(rain_lags) + 2
        self.rain_gate_enable = bool(rain_gate_enable)
        if self.rain_gate_enable:
            self.rain_gate = RainFiLMGate(
                rain_dim=self.rain_ctx_dim,
                d_model=d_model,
                hidden_dim=max(8, int(rain_gate_hidden)),
                scale=rain_gate_scale,
            )
        else:
            self.rain_gate = None
    def _batched_edge_index(self, batch_size: int, device: torch.device):
        key = (int(batch_size), str(device))
        if key in self._edge_cache:
            return self._edge_cache[key]
        e = self.edge_index_base.to(device)
        offsets = (torch.arange(batch_size, device=device, dtype=torch.long) * self.n_nodes).view(batch_size, 1, 1)
        eb = e.view(1, 2, -1) + offsets
        eb = eb.permute(1, 0, 2).reshape(2, -1).contiguous()
        self._edge_cache[key] = eb
        return eb
    def forward(self, x_graph: torch.Tensor):
        b, n, _ = x_graph.shape
        if n != self.n_nodes:
            raise RuntimeError(f"node size mismatch: got {n}, expected {self.n_nodes}")
        if self.node_emb is not None:
            node_ids = self.node_index_base.to(x_graph.device).view(1, n).expand(b, n)
            emb = self.node_emb(node_ids)
            x = torch.cat([x_graph, emb], dim=-1)
        else:
            x = x_graph
        h = self.input_proj(x)
        if self.rain_gate is not None:
            rain_end = self.rain_start + self.rain_ctx_dim
            if rain_end > x_graph.shape[2]:
                raise RuntimeError(
                    f"invalid rain gate layout: rain_start={self.rain_start} rain_end={rain_end} total_dim={x_graph.shape[2]}"
                )
            rain_ctx = x_graph[:, 0, self.rain_start:rain_end]
            h = self.rain_gate(h, rain_ctx)
        h = h.reshape(b * n, -1)
        edge_index = self._batched_edge_index(b, x_graph.device)
        for layer in self.layers:
            h = layer(h, edge_index)
        out = self.head(h).reshape(b, n, self.out_dim)
        main_delta = out[:, :, 0]
        aux_delta = out[:, :, 1:] if self.n_aux_targets > 0 else None
        return main_delta, aux_delta


def make_loaders(
    x_train_g,
    y_train_g,
    x_val_g,
    y_val_g,
    batch_size: int,
    y_train_aux_g=None,
    y_val_aux_g=None,


):
    if y_train_aux_g is not None:
        train_ds = TensorDataset(
            torch.from_numpy(x_train_g),
            torch.from_numpy(y_train_g),
            torch.from_numpy(y_train_aux_g),
        )
    else:
        train_ds = TensorDataset(torch.from_numpy(x_train_g), torch.from_numpy(y_train_g))
    if y_val_aux_g is not None:
        val_ds = TensorDataset(
            torch.from_numpy(x_val_g),
            torch.from_numpy(y_val_g),
            torch.from_numpy(y_val_aux_g),
        )
    else:
        val_ds = TensorDataset(torch.from_numpy(x_val_g), torch.from_numpy(y_val_g))
    pin = DEVICE == "cuda"
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin,
        drop_last=False,
    )
    return train_loader, val_loader


def compute_physics_losses(
    pred: torch.Tensor,
    x_graph: torch.Tensor,
    model: PhysicsInformedGNN,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    extra_lags: int,
):
    b, n = pred.shape
    pred_flat = pred.reshape(b * n)
    edge_index = model._batched_edge_index(b, pred.device)
    src, dst = edge_index
    smooth = torch.mean((pred_flat[src] - pred_flat[dst]) ** 2)
    extra_dim = len(extra_cols) * max(0, int(extra_lags))
    fixed_dim = 1 + int(y_lags) + int(extra_dim) + (int(rain_lags) + 1) + 1
    static_dim = int(x_graph.shape[2]) - fixed_dim
    if static_dim < 0:
        raise RuntimeError(
            f"invalid feature layout: total_dim={int(x_graph.shape[2])} fixed_dim={fixed_dim} static_dim={static_dim}"
        )
    y_start = 1 + static_dim
    if y_lags >= 2:
        y_prev = x_graph[:, :, y_start]
        y_prev2 = x_graph[:, :, y_start + 1]
        prev_delta = y_prev - y_prev2
        dyn = nn.functional.mse_loss(pred, prev_delta)
    else:
        dyn = torch.zeros((), device=pred.device, dtype=pred.dtype)
    rain_start = y_start + int(y_lags) + int(extra_dim)
    rain_now = x_graph[:, 0, rain_start]
    mass = torch.mean((pred.mean(dim=1) - model.beta_rain * rain_now) ** 2)
    return smooth, mass, dyn


def compute_rollout_loss_batch(
    model: PhysicsInformedGNN,
    y_hist: torch.Tensor,
    y_future: torch.Tensor,
    extra_hist: torch.Tensor | None,
    extra_future: torch.Tensor | None,
    rain_steps: torch.Tensor,
    rain_mean: torch.Tensor,
    scaler_mean_t: torch.Tensor,
    scaler_std_t: torch.Tensor,
    static_feat_t: torch.Tensor | None,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
):
    b, _, n_nodes = y_hist.shape
    device = y_hist.device
    dtype = y_hist.dtype
    node_pos = torch.arange(n_nodes, device=device, dtype=dtype) / max(1.0, float(n_nodes - 1))
    node_pos = node_pos.view(1, n_nodes, 1).expand(b, n_nodes, 1)
    static_part = None
    if static_feat_t is not None and static_feat_t.shape[1] > 0:
        static_part = static_feat_t.to(device=device, dtype=dtype).unsqueeze(0).expand(b, -1, -1)
    joint_aux_map = {col: idx for idx, col in enumerate(joint_aux_cols)}
    y_hist_cur = y_hist
    extra_hist_cur = extra_hist
    losses = []
    horizon = int(y_future.shape[1])
    for step in range(horizon):
        feat_parts = [node_pos]
        if static_part is not None:
            feat_parts.append(static_part)
        feat_parts.append(y_hist_cur.permute(0, 2, 1))
        if extra_hist_cur is not None and extra_lags > 0:
            extra_parts = [extra_hist_cur[:, col_idx].permute(0, 2, 1) for col_idx in range(extra_hist_cur.shape[1])]
            if extra_parts:
                feat_parts.append(torch.cat(extra_parts, dim=-1))
        step_r = rain_steps[:, step].unsqueeze(1).expand(b, n_nodes, rain_lags + 1)
        step_mean = rain_mean.view(b, 1, 1).expand(b, n_nodes, 1)
        feat_parts.extend([step_r, step_mean])
        x_step = torch.cat(feat_parts, dim=-1)
        x_step = apply_scaler_torch(x_step, scaler_mean_t, scaler_std_t)
        pred_delta, pred_aux = model(x_step)
        y_next = y_hist_cur[:, 0, :] + pred_delta
        y_next = torch.nan_to_num(y_next, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
        y_next = y_next.clamp(-SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP)
        losses.append(nn.functional.mse_loss(y_next, y_future[:, step, :]))
        y_hist_cur = torch.cat([y_next.unsqueeze(1), y_hist_cur[:, :-1, :]], dim=1)
        if extra_hist_cur is not None and extra_lags > 0:
            next_cols = []
            for col_idx, col in enumerate(extra_cols):
                prev_ex = extra_hist_cur[:, col_idx, 0, :]
                if col in joint_aux_map and pred_aux is not None:
                    ex_next = prev_ex + pred_aux[:, :, joint_aux_map[col]]
                elif extra_future is not None:
                    ex_next = extra_future[:, col_idx, step, :]
                else:
                    ex_next = prev_ex
                ex_next = torch.nan_to_num(ex_next, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
                ex_next = ex_next.clamp(-SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP)
                next_cols.append(torch.cat([ex_next.unsqueeze(1), extra_hist_cur[:, col_idx, :-1, :]], dim=1))
            extra_hist_cur = torch.stack(next_cols, dim=1)
    return torch.stack(losses).mean() if losses else torch.zeros((), device=device, dtype=dtype)


def evaluate_rmse(model, loader):
    model.eval()
    sse = 0.0
    cnt = 0
    with torch.no_grad():
        for batch in loader:
            xb = batch[0].to(DEVICE, non_blocking=True)
            yb = batch[1].to(DEVICE, non_blocking=True)
            pred, _ = model(xb)
            diff = pred - yb
            sse += float(torch.sum(diff * diff).item())
            cnt += int(yb.numel())
    if cnt == 0:
        return float("inf")
    return float(np.sqrt(sse / cnt))


def train_one_fold(
    x_train,
    y_train,
    y_train_aux,
    x_val,
    y_val,
    y_val_aux,
    fold_idx: int,
    n_nodes: int,
    edge_index_np: np.ndarray,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
    epoch_oof_eval_every: int = 0,
    epoch_oof_eval_fn=None,
    aux_loss_weight: float = 1.0,
    rollout_train_data=None,
    scaler_mean: np.ndarray | None = None,
    scaler_std: np.ndarray | None = None,
    static_feat: np.ndarray | None = None,
):
    x_train_g, y_train_g = flat_to_graph_samples(x_train, y_train, n_nodes, tag=f"fold{fold_idx+1}:train")
    x_val_g, y_val_g = flat_to_graph_samples(x_val, y_val, n_nodes, tag=f"fold{fold_idx+1}:val")
    n_aux = len(joint_aux_cols)
    y_train_aux_g = flat_to_graph_aux_samples(y_train_aux, n_nodes, n_aux, tag=f"fold{fold_idx+1}:train")
    y_val_aux_g = flat_to_graph_aux_samples(y_val_aux, n_nodes, n_aux, tag=f"fold{fold_idx+1}:val")
    train_loader, val_loader = make_loaders(
        x_train_g,
        y_train_g,
        x_val_g,
        y_val_g,
        GRAPH_BATCH_SIZE,
        y_train_aux_g=y_train_aux_g,
        y_val_aux_g=y_val_aux_g,
    )
    model = PhysicsInformedGNN(
        n_features=x_train_g.shape[2],
        n_nodes=n_nodes,
        edge_index_np=edge_index_np,
        d_model=GNN_D_MODEL,
        n_layers=GNN_N_LAYERS,
        dropout=GNN_DROPOUT,
        node_embed_dim=NODE_EMBED_DIM,
        n_aux_targets=n_aux,
        layer_type=GNN_LAYER_TYPE,
        attn_dropout=ATTN_DROPOUT,
        attn_use_out_proj=ATTN_USE_OUT_PROJ,
        y_lags=y_lags,
        rain_lags=rain_lags,
        n_extra_cols=len(extra_cols),
        extra_lags=extra_lags,
        static_dim=0 if static_feat is None else int(static_feat.shape[1]),
        rain_gate_enable=RAIN_GATE_ENABLE,
        rain_gate_hidden=RAIN_GATE_HIDDEN,
        rain_gate_scale=RAIN_GATE_SCALE,
    ).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=LR_FACTOR,
        patience=LR_PATIENCE,
        min_lr=MIN_LR,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    best_rmse = float("inf")
    best_state = None
    best_epoch_oof_std = float("inf")
    best_oof_state = None
    patience_left = PATIENCE
    rollout_enabled = (
        ROLLOUT_LOSS_ENABLE
        and rollout_train_data is not None
        and int(rollout_train_data.get("count", 0)) > 0
        and scaler_mean is not None
        and scaler_std is not None
    )
    if rollout_enabled:
        scaler_mean_t = torch.from_numpy(scaler_mean.astype(np.float32, copy=False)).to(DEVICE)
        scaler_std_t = torch.from_numpy(scaler_std.astype(np.float32, copy=False)).to(DEVICE)
        static_feat_t = None if static_feat is None else torch.from_numpy(static_feat.astype(np.float32, copy=False)).to(DEVICE)
    else:
        scaler_mean_t = None
        scaler_std_t = None
        static_feat_t = None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_sse = 0.0
        train_cnt = 0
        rollout_loss_mean = float("nan")
        for batch in train_loader:
            xb = batch[0].to(DEVICE, non_blocking=True)
            yb = batch[1].to(DEVICE, non_blocking=True)
            yb_aux = batch[2].to(DEVICE, non_blocking=True) if len(batch) >= 3 else None
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred, pred_aux = model(xb)
                loss_main = nn.functional.mse_loss(pred, yb)
                if yb_aux is not None and pred_aux is not None:
                    loss_aux = nn.functional.mse_loss(pred_aux, yb_aux)
                else:
                    loss_aux = torch.zeros((), device=pred.device, dtype=pred.dtype)
                loss_smooth, loss_mass, loss_dyn = compute_physics_losses(
                    pred=pred,
                    x_graph=xb,
                    model=model,
                    y_lags=y_lags,
                    rain_lags=rain_lags,
                    extra_cols=extra_cols,
                    extra_lags=extra_lags,
                )
                loss = (
                    loss_main
                    + float(aux_loss_weight) * loss_aux
                    + float(LAMBDA_SMOOTH) * loss_smooth
                    + float(LAMBDA_MASS) * loss_mass
                    + float(LAMBDA_DYN) * loss_dyn
                )
            scaler.scale(loss).backward()
            if GRAD_CLIP_NORM > 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(opt)
            scaler.update()
            diff = pred.detach() - yb
            train_sse += float(torch.sum(diff * diff).item())
            train_cnt += int(yb.numel())
        if rollout_enabled:
            perm = np.random.permutation(int(rollout_train_data["count"]))
            rollout_loss_sum = 0.0
            rollout_batches = 0
            for st in range(0, len(perm), ROLLOUT_BATCH_SIZE):
                idx = perm[st : st + ROLLOUT_BATCH_SIZE]
                y_hist = torch.from_numpy(rollout_train_data["y_hist"][idx]).to(DEVICE, non_blocking=True)
                y_future = torch.from_numpy(rollout_train_data["y_future"][idx]).to(DEVICE, non_blocking=True)
                rain_steps = torch.from_numpy(rollout_train_data["rain_steps"][idx]).to(DEVICE, non_blocking=True)
                rain_mean = torch.from_numpy(rollout_train_data["rain_mean"][idx]).to(DEVICE, non_blocking=True)
                extra_hist = None
                extra_future = None
                if rollout_train_data.get("extra_hist") is not None:
                    extra_hist = torch.from_numpy(rollout_train_data["extra_hist"][idx]).to(DEVICE, non_blocking=True)
                    extra_future = torch.from_numpy(rollout_train_data["extra_future"][idx]).to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    rollout_loss = compute_rollout_loss_batch(
                        model=model,
                        y_hist=y_hist,
                        y_future=y_future,
                        extra_hist=extra_hist,
                        extra_future=extra_future,
                        rain_steps=rain_steps,
                        rain_mean=rain_mean,
                        scaler_mean_t=scaler_mean_t,
                        scaler_std_t=scaler_std_t,
                        static_feat_t=static_feat_t,
                        y_lags=y_lags,
                        rain_lags=rain_lags,
                        extra_cols=extra_cols,
                        joint_aux_cols=joint_aux_cols,
                        extra_lags=extra_lags,
                    )
                    loss_rollout = float(ROLLOUT_WEIGHT) * rollout_loss
                scaler.scale(loss_rollout).backward()
                if GRAD_CLIP_NORM > 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(opt)
                scaler.update()
                rollout_loss_sum += float(rollout_loss.item())
                rollout_batches += 1
            rollout_loss_mean = rollout_loss_sum / max(rollout_batches, 1)
        train_rmse = float(np.sqrt(train_sse / max(train_cnt, 1)))
        val_rmse = evaluate_rmse(model, val_loader)
        scheduler.step(val_rmse)
        cur_lr = float(opt.param_groups[0]["lr"])
        std = float(STD_DEV[TARGET])
        rollout_log = f" rollout_mse={rollout_loss_mean:.6f}" if np.isfinite(rollout_loss_mean) else ""
        log(
            f"fold={fold_idx+1} epoch={epoch}/{EPOCHS} "
            f"train_rmse={train_rmse:.6f} val_rmse={val_rmse:.6f} val_std={val_rmse/std:.6f} "
            f"lr={cur_lr:.2e}{rollout_log}"
        )
        if epoch_oof_eval_fn is not None and epoch_oof_eval_every > 0 and (epoch % epoch_oof_eval_every == 0):
            try:
                epoch_oof_std = float(epoch_oof_eval_fn(model))
                log(
                    f"fold={fold_idx+1} epoch={epoch}/{EPOCHS} "
                    f"oof_lb_std@epoch={epoch_oof_std:.6f}"
                )
                if np.isfinite(epoch_oof_std) and epoch_oof_std < best_epoch_oof_std:
                    best_epoch_oof_std = epoch_oof_std
                    best_oof_state = copy.deepcopy(model.state_dict())
            except Exception as exc:
                log(f"fold={fold_idx+1} epoch={epoch}/{EPOCHS} oof_lb_eval_skipped: {exc}")
        if val_rmse < best_rmse:
            best_rmse = val_rmse
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                log(f"fold={fold_idx+1} early_stop")
                break
    if best_state is None:
        raise RuntimeError("best_state is None (training failed?)")

    model.load_state_dict(best_state)
    return {
        "model": model,
        "best_val_rmse": float(best_rmse),
        "best_val_state": best_state,
        "best_epoch_oof_std": float(best_epoch_oof_std) if np.isfinite(best_epoch_oof_std) else float("nan"),
        "best_oof_state": best_oof_state,
    }


def predict_delta(model, x_np: np.ndarray, batch_size: int = 64):
    # x_np: [N, F] or [B, N, F]
    if x_np.ndim == 2:
        x_np = x_np[None, :, :]
        squeeze = True
    else:
        squeeze = False
    model.eval()
    out = np.empty((x_np.shape[0], x_np.shape[1]), dtype=np.float32)
    out_aux = None
    with torch.no_grad():
        for i in range(0, x_np.shape[0], batch_size):
            xb = torch.from_numpy(x_np[i : i + batch_size]).to(DEVICE)
            yp, yp_aux = model(xb)
            yp = yp.detach().cpu().numpy().astype(np.float32, copy=False)
            out[i : i + batch_size, :] = yp
            if yp_aux is not None:
                yp_aux = yp_aux.detach().cpu().numpy().astype(np.float32, copy=False)
                if out_aux is None:
                    out_aux = np.empty((x_np.shape[0], x_np.shape[1], yp_aux.shape[2]), dtype=np.float32)
                out_aux[i : i + batch_size, :, :] = yp_aux
    if squeeze:
        return out[0], (out_aux[0] if out_aux is not None else None)
    return out, out_aux


def predict_event_recursive(
    ev,
    model,
    scaler_mean,
    scaler_std,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
    static_feat: np.ndarray | None,
):
    t = ev["t"]
    y = ev["y"].copy()
    rain = ev["rain"]
    n = min(len(t), y.shape[0], len(rain))
    t = t[:n]
    y = y[:n]
    rain = rain[:n]
    extra = {k: v[:n] for k, v in ev["extra"].items()}
    rain_hist = build_rain_lags(rain, rain_lags)
    n_nodes = y.shape[1]
    node_pos = np.arange(n_nodes, dtype=np.float32) / max(1.0, float(n_nodes - 1))
    rain_mean = float(np.mean(rain)) if n > 0 else 0.0
    for tt in range(start_t, n):
        step_y = np.stack([y[tt - 1 - k, :] for k in range(y_lags)], axis=1).astype(
            np.float32, copy=False
        )
        step_extra = []
        if extra_lags > 0:
            for col in extra_cols:
                ex = extra[col]
                ex_lag = np.stack(
                    [ex[tt - 1 - k, :] for k in range(extra_lags)],
                    axis=1,
                ).astype(np.float32, copy=False)
                step_extra.append(ex_lag)
        step_r = np.tile(rain_hist[tt, : rain_lags + 1][None, :], (n_nodes, 1)).astype(
            np.float32, copy=False
        )
        step_mean = np.full((n_nodes, 1), rain_mean, dtype=np.float32)
        feat_parts = [node_pos[:, None]]
        if static_feat is not None and static_feat.shape[1] > 0:
            feat_parts.append(static_feat)
        feat_parts.append(step_y)
        if step_extra:
            feat_parts.append(np.concatenate(step_extra, axis=1))
        feat_parts.extend([step_r, step_mean])
        feat = np.concatenate(feat_parts, axis=1)
        np.nan_to_num(feat, copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
        np.clip(feat, -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=feat)
        feat = apply_scaler(feat, scaler_mean, scaler_std)
        delta, delta_aux = predict_delta(model, feat, batch_size=max(1, PRED_BATCH_SIZE // max(1, n_nodes)))
        np.nan_to_num(delta, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        np.clip(delta, -SAFE_DELTA_CLIP, SAFE_DELTA_CLIP, out=delta)
        y[tt, :] = y[tt - 1, :] + delta
        np.nan_to_num(y[tt, :], copy=False, nan=0.0, posinf=SAFE_LEVEL_CLIP, neginf=-SAFE_LEVEL_CLIP)
        np.clip(y[tt, :], -SAFE_LEVEL_CLIP, SAFE_LEVEL_CLIP, out=y[tt, :])
        if delta_aux is not None:
            for aux_idx, col in enumerate(joint_aux_cols):
                ex = extra[col]
                ex[tt, :] = ex[tt - 1, :] + delta_aux[:, aux_idx]
                np.nan_to_num(ex[tt, :], copy=False, nan=0.0, posinf=SAFE_FEATURE_CLIP, neginf=-SAFE_FEATURE_CLIP)
                np.clip(ex[tt, :], -SAFE_FEATURE_CLIP, SAFE_FEATURE_CLIP, out=ex[tt, :])
    return {"t": t, "pred": y, "n": n}


def evaluate_oof_lb_equivalent(
    events,
    model,
    scaler_mean,
    scaler_std,
    target,
    start_t: int,
    y_lags: int,
    rain_lags: int,
    extra_cols,
    joint_aux_cols,
    extra_lags: int,
    static_feat: np.ndarray | None,
):
    # Competition metric equivalent for this single target: event-wise mean over nodes of standardized RMSE.
    if OOF_LB_MAX_EVENTS > 0:
        eval_events = events[:OOF_LB_MAX_EVENTS]
    else:
        eval_events = events
    std = float(STD_DEV[target])
    event_scores = []
    node_scores = []
    total = len(eval_events)
    for j, ev in enumerate(eval_events, 1):
        out = predict_event_recursive(
            ev=ev,
            model=model,
            scaler_mean=scaler_mean,
            scaler_std=scaler_std,
            start_t=start_t,
            y_lags=y_lags,
            rain_lags=rain_lags,
            extra_cols=extra_cols,
            joint_aux_cols=joint_aux_cols,
            extra_lags=extra_lags,
            static_feat=static_feat,
        )
        n = out["n"]
        if n <= start_t:
            continue
        y_true = ev["y"][start_t:n, :]
        y_pred = out["pred"][start_t:n, :]
        err = y_pred - y_true
        rmse_node = np.sqrt(np.mean(err * err, axis=0))
        std_node = (rmse_node / std).astype(np.float32, copy=False)
        event_score = float(np.mean(std_node))
        event_scores.append(event_score)
        node_scores.append(std_node)
        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
            log(f"oof-lb eval progress: {j}/{total} event_std_rmse={event_score:.6f}")
    if not event_scores:
        return float("nan"), None
    node_mean = np.mean(np.stack(node_scores, axis=0), axis=0).astype(np.float32)
    return float(np.mean(event_scores)), node_mean


def build_prediction_df(model_id: int, node_type: int, events, node_ids, fold_states, static_feat: np.ndarray | None):
    # fold-average prediction
    fold_parts = []
    for fold_idx, st in enumerate(fold_states):
        model = st["model"]
        scaler_mean = st["scaler_mean"]
        scaler_std = st["scaler_std"]
        rows = []
        total = len(events)
        for j, ev in enumerate(events, 1):
            out = predict_event_recursive(
                ev=ev,
                model=model,
                scaler_mean=scaler_mean,
                scaler_std=scaler_std,
                start_t=START_T,
                y_lags=Y_LAGS,
                rain_lags=RAIN_LAGS,
                extra_cols=EXTRA_COLS_ACTIVE,
                joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
                extra_lags=EXTRA_LAGS,
                static_feat=static_feat,
            )
            n = out["n"]
            ts = out["t"][START_T:n]
            pred = out["pred"][START_T:n, :]
            if ts.size == 0:
                continue
            part = pd.DataFrame(
                {
                    "model_id": np.full((ts.size * pred.shape[1],), model_id, dtype=np.int8),
                    "event_id": np.full((ts.size * pred.shape[1],), ev["event_id"], dtype=np.int32),
                    "node_type": np.full((ts.size * pred.shape[1],), node_type, dtype=np.int8),
                    "node_id": np.tile(node_ids, ts.size),
                    "timestep": np.repeat(ts.astype(np.int32), pred.shape[1]),
                    "water_level": pred.reshape(-1).astype(np.float32, copy=False),
                }
            )
            rows.append(part)
            if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
                log(f"predict fold={fold_idx+1} progress: {j}/{total} events")
        if rows:
            fold_parts.append(pd.concat(rows, ignore_index=True))
        gc.collect()
    if not fold_parts:
        return pd.DataFrame(columns=KEY_COLS + ["water_level"])
    pred = pd.concat(fold_parts, ignore_index=True)
    pred = pred.groupby(KEY_COLS, as_index=False)["water_level"].mean()
    return pred


def build_oof_prediction_df(model_id: int, node_type: int, train_events_all, node_ids, fold_states, static_feat: np.ndarray | None):
    rows = []
    for i, st in enumerate(fold_states, 1):
        val_ids = st["val_ids"]
        va_events = []
        for eid in val_ids:
            ev = train_events_all[eid]
            va_events.append(
                {
                    "event_id": ev["event_id"],
                    "t": ev["t"],
                    "y": ev["y"].copy(),
                    "rain": ev["rain"],
                    "extra": {k: v.copy() for k, v in ev["extra"].items()},
                }
            )
        fill_missing(va_events, st["node_means"], st["extra_means"])
        total = len(va_events)
        for j, ev in enumerate(va_events, 1):
            out = predict_event_recursive(
                ev=ev,
                model=st["model"],
                scaler_mean=st["scaler_mean"],
                scaler_std=st["scaler_std"],
                start_t=START_T,
                y_lags=Y_LAGS,
                rain_lags=RAIN_LAGS,
                extra_cols=EXTRA_COLS_ACTIVE,
                joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
                extra_lags=EXTRA_LAGS,
                static_feat=static_feat,
            )
            n = out["n"]
            ts = out["t"][START_T:n]
            pred = out["pred"][START_T:n, :]
            y_true = ev["y"][START_T:n, :]
            if ts.size == 0:
                continue
            m = min(ts.size, pred.shape[0], y_true.shape[0])
            if m <= 0:
                continue
            ts = ts[:m]
            pred = pred[:m, :]
            y_true = y_true[:m, :]
            part = pd.DataFrame(
                {
                    "model_id": np.full((m * pred.shape[1],), model_id, dtype=np.int8),
                    "event_id": np.full((m * pred.shape[1],), ev["event_id"], dtype=np.int32),
                    "node_type": np.full((m * pred.shape[1],), node_type, dtype=np.int8),
                    "node_id": np.tile(node_ids, m),
                    "timestep": np.repeat(ts.astype(np.int32), pred.shape[1]),
                    "water_level_true": y_true.reshape(-1).astype(np.float32, copy=False),
                    "water_level_pred": pred.reshape(-1).astype(np.float32, copy=False),
                }
            )
            rows.append(part)
            if j % max(1, LOG_EVERY_EVENTS) == 0 or j == total:
                log(f"oof table fold={st['fold_idx']+1} progress: {j}/{total} events")
        del va_events
        gc.collect()
    if not rows:
        return pd.DataFrame(columns=KEY_COLS + ["water_level_true", "water_level_pred"])
    oof = pd.concat(rows, ignore_index=True)
    oof = oof.sort_values(KEY_COLS).reset_index(drop=True)
    return oof


def build_submission(sample_path: Path, pred_df: pd.DataFrame, output_path: str):
    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sample = pd.read_csv(sample_path)
    sub = sample.copy()
    if "water_level" not in sub.columns:
        sub["water_level"] = np.nan
    if pred_df is None:
        pred_df = pd.DataFrame(columns=KEY_COLS + ["water_level"])
    # Fast path: full-length prediction table
    if len(pred_df) > 0 and len(sample) == len(pred_df) and "water_level" in pred_df.columns:
        sub["water_level"] = pd.to_numeric(pred_df["water_level"], errors="coerce").to_numpy(dtype=np.float32, copy=False)
    elif len(pred_df) > 0 and "water_level" in pred_df.columns:
        # Partial merge path (for target-subset predictions)
        if "timestep" in sub.columns:
            sub["__step_idx"] = sub["timestep"].astype(np.int32)
        else:
            sub["__step_idx"] = sub.groupby(["model_id", "event_id", "node_type", "node_id"]).cumcount()
        pred = pred_df.copy()
        if "timestep" in pred.columns:
            pred["__step_idx"] = pred["timestep"].astype(np.int32)
        else:
            pred["__step_idx"] = pred.groupby(["model_id", "event_id", "node_type", "node_id"]).cumcount()
        merge_cols = ["model_id", "event_id", "node_type", "node_id", "__step_idx", "water_level"]
        keep_cols = [c for c in merge_cols if c in pred.columns]
        pred = pred[keep_cols]
        sub = sub.merge(
            pred,
            on=["model_id", "event_id", "node_type", "node_id", "__step_idx"],
            how="left",
            suffixes=("_base", "_pred"),
        )
        if "water_level_pred" in sub.columns:
            sub["water_level"] = sub["water_level_pred"].where(
                sub["water_level_pred"].notna(),
                pd.to_numeric(sub.get("water_level_base"), errors="coerce"),
            )
            sub = sub.drop(columns=["water_level_base", "water_level_pred"], errors="ignore")
        elif "water_level" not in sub.columns:
            # Last-resort fallback: keep original sample column
            sub["water_level"] = pd.to_numeric(sample.get("water_level"), errors="coerce")
    else:
        print("warning: pred_df empty or missing water_level; using fallback fill", flush=True)
    wl = pd.to_numeric(sub["water_level"], errors="coerce")
    missing = int(wl.isna().sum())
    if missing > 0:
        pred_mean = np.nan
        if len(pred_df) > 0 and "water_level" in pred_df.columns:
            pred_vals = pd.to_numeric(pred_df["water_level"], errors="coerce").to_numpy(dtype=np.float64, copy=False)
            if np.isfinite(pred_vals).any():
                pred_mean = float(np.nanmean(pred_vals))
        base_vals = pd.to_numeric(sample.get("water_level"), errors="coerce").to_numpy(dtype=np.float64, copy=False)
        base_mean = float(np.nanmean(base_vals)) if np.isfinite(base_vals).any() else np.nan
        fill_value = pred_mean if np.isfinite(pred_mean) else (base_mean if np.isfinite(base_mean) else 0.0)
        wl = wl.fillna(fill_value)
        print(
            f"warning: filled missing submission rows with fallback value={fill_value:.6f} rows={missing}",
            flush=True,
        )
    sub["water_level"] = wl.astype(np.float32, copy=False)
    sub = sub.drop(columns=["__step_idx"], errors="ignore")
    sub.to_csv(out_path, index=False)


# =========================
# Main
# =========================


def main():
    t0 = time.time()
    set_seed(SEED)
    model_id, node_type = TARGET
    models_root = resolve_models_root(DATASET_ROOT)
    sample_path = resolve_sample_submission(SAMPLE_SUBMISSION_PATH, models_root)
    mdir = model_dir(models_root, model_id)
    train_ids = list_event_ids(mdir, "train")
    test_ids = list_event_ids(mdir, "test")
    if MAX_TRAIN_EVENTS > 0:
        train_ids = train_ids[:MAX_TRAIN_EVENTS]
    if MAX_TEST_EVENTS > 0:
        test_ids = test_ids[:MAX_TEST_EVENTS]
    if len(train_ids) < N_FOLDS:
        raise RuntimeError(f"not enough train events ({len(train_ids)}) for N_FOLDS={N_FOLDS}")
    rain_col = detect_rain_col(mdir, "train", train_ids[0])
    node_ids = load_node_ids(mdir, node_type)
    static_feat, static_cols_active = load_static_node_features(
        mdir=mdir,
        node_type=node_type,
        node_ids=node_ids,
        requested_cols=STATIC_COLS_REQ,
        standardize=STATIC_STANDARDIZE,
    )
    if not STATIC_FEATURES_ENABLE:
        static_feat = np.empty((len(node_ids), 0), dtype=np.float32)
        static_cols_active = []
    global EXTRA_COLS_ACTIVE, JOINT_AUX_COLS_ACTIVE
    requested_extra_cols = []
    for c in (EXTRA_COLS_REQ + JOINT_AUX_COLS_REQ):
        if c and c not in requested_extra_cols:
            requested_extra_cols.append(c)
    EXTRA_COLS_ACTIVE = resolve_extra_cols(
        mdir=mdir,
        split="train",
        event_id=train_ids[0],
        node_type=node_type,
        requested_cols=requested_extra_cols,
    )
    JOINT_AUX_COLS_ACTIVE = [c for c in JOINT_AUX_COLS_REQ if c in EXTRA_COLS_ACTIVE]
    log(f"DEVICE={DEVICE} USE_AMP={USE_AMP}")
    log(f"TARGET={TARGET} node_count={len(node_ids)}")
    log(f"train_events={len(train_ids)} test_events={len(test_ids)}")
    log(
        f"N_FOLDS={N_FOLDS} RUN_SINGLE_FOLD={RUN_SINGLE_FOLD} FOLD_INDEX={FOLD_INDEX} "
        f"EPOCHS={EPOCHS} GRAPH_BATCH_SIZE={GRAPH_BATCH_SIZE} "
        f"MAX_TRAIN_ROWS={MAX_TRAIN_ROWS} MAX_VAL_ROWS={MAX_VAL_ROWS}"
    )
    log(
        f"Y_LAGS={Y_LAGS} RAIN_LAGS={RAIN_LAGS} GNN_D_MODEL={GNN_D_MODEL} "
        f"GNN_N_LAYERS={GNN_N_LAYERS} GNN_DROPOUT={GNN_DROPOUT} NODE_EMBED_DIM={NODE_EMBED_DIM} "
        f"GNN_LAYER_TYPE={GNN_LAYER_TYPE} ATTN_DROPOUT={ATTN_DROPOUT} ATTN_USE_OUT_PROJ={ATTN_USE_OUT_PROJ} "
        f"RAIN_GATE_ENABLE={RAIN_GATE_ENABLE} RAIN_GATE_HIDDEN={RAIN_GATE_HIDDEN} RAIN_GATE_SCALE={RAIN_GATE_SCALE} "
        f"ROLLOUT_LOSS_ENABLE={ROLLOUT_LOSS_ENABLE} ROLLOUT_STEPS={ROLLOUT_STEPS} "
        f"ROLLOUT_WEIGHT={ROLLOUT_WEIGHT} ROLLOUT_BATCH_SIZE={ROLLOUT_BATCH_SIZE} ROLLOUT_MAX_SAMPLES={ROLLOUT_MAX_SAMPLES}"
    )
    log(
        f"LAMBDA_SMOOTH={LAMBDA_SMOOTH} LAMBDA_MASS={LAMBDA_MASS} LAMBDA_DYN={LAMBDA_DYN}"
    )
    log(f"STATIC_FEATURES_ENABLE={STATIC_FEATURES_ENABLE} STATIC_COLS_ACTIVE={static_cols_active} STATIC_DIM={int(static_feat.shape[1])}")
    log(f"EXTRA_COLS_ACTIVE={EXTRA_COLS_ACTIVE} EXTRA_LAGS={EXTRA_LAGS}")
    log(
        f"JOINT_AUX_COLS_ACTIVE={JOINT_AUX_COLS_ACTIVE} AUX_LOSS_WEIGHT={AUX_LOSS_WEIGHT}"
    )
    log(f"SAVE_BUNDLE={SAVE_BUNDLE} BUNDLE_DIR={BUNDLE_DIR}")
    log(
        f"LR={LR} MIN_LR={MIN_LR} LR_FACTOR={LR_FACTOR} LR_PATIENCE={LR_PATIENCE} "
        f"PATIENCE={PATIENCE} GRAD_CLIP_NORM={GRAD_CLIP_NORM}"
    )
    log(
        f"OOF_LB_EVAL={OOF_LB_EVAL} OOF_LB_MAX_EVENTS={OOF_LB_MAX_EVENTS} "
        f"EPOCH_OOF_EVAL_EVERY={EPOCH_OOF_EVAL_EVERY} EPOCH_OOF_MAX_EVENTS={EPOCH_OOF_MAX_EVENTS}"
    )
    edge_index_np = load_1d_edge_index(mdir, node_ids)
    log(f"graph edge_index loaded: edges={edge_index_np.shape[1]} (undirected expanded)")
    log("Step 1: preload events")
    train_events_all = {}
    for j, eid in enumerate(train_ids, 1):
        train_events_all[eid] = load_event(mdir, "train", eid, node_type, node_ids, TARGET_COL, rain_col)
        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == len(train_ids):
            log(f"preload train: {j}/{len(train_ids)}")
    test_events_all = {}
    for j, eid in enumerate(test_ids, 1):
        test_events_all[eid] = load_event(mdir, "test", eid, node_type, node_ids, TARGET_COL, rain_col)
        if j % max(1, LOG_EVERY_EVENTS) == 0 or j == len(test_ids):
            log(f"preload test: {j}/{len(test_ids)}")
    folds = split_events_kfold(train_ids, N_FOLDS, EVENT_SPLIT_SEED + model_id * 1000 + node_type)
    if RUN_SINGLE_FOLD:
        if not (0 <= FOLD_INDEX < N_FOLDS):
            raise RuntimeError(f"FOLD_INDEX out of range: {FOLD_INDEX} for N_FOLDS={N_FOLDS}")
        fold_indices = [FOLD_INDEX]
    else:
        fold_indices = list(range(N_FOLDS))
    log(f"fold sizes={[len(f) for f in folds]} active_folds={[i+1 for i in fold_indices]}")
    fold_states = []
    sample_val_scores = []
    oof_lb_scores = []
    log("Step 2: fold training")
    for run_i, fold_idx in enumerate(fold_indices, 1):
        val_ids = folds[fold_idx]
        tr_ids = fold_train_events(folds, fold_idx)
        log(
            f"fold {fold_idx+1}/{N_FOLDS} (run {run_i}/{len(fold_indices)}): "
            f"train_events={len(tr_ids)} val_events={len(val_ids)}"
        )
        tr_events = [train_events_all[eid] for eid in tr_ids]
        va_events = [train_events_all[eid] for eid in val_ids]
        means = compute_node_means_from_events(tr_events)
        extra_means = compute_extra_means_from_events(tr_events, EXTRA_COLS_ACTIVE)
        # copy refs, then fill in-place for this fold only
        tr_events = [
            {
                "event_id": ev["event_id"],
                "t": ev["t"],
                "y": ev["y"].copy(),
                "rain": ev["rain"],
                "extra": {k: v.copy() for k, v in ev["extra"].items()},
            }
            for ev in tr_events
        ]
        va_events = [
            {
                "event_id": ev["event_id"],
                "t": ev["t"],
                "y": ev["y"].copy(),
                "rain": ev["rain"],
                "extra": {k: v.copy() for k, v in ev["extra"].items()},
            }
            for ev in va_events
        ]
        fill_missing(tr_events, means, extra_means)
        fill_missing(va_events, means, extra_means)
        seed_base = SEED + fold_idx * 100
        x_train, y_train, y_train_aux, rows_pos_train, rows_kept_train = build_samples(
            tr_events,
            start_t=START_T,
            y_lags=Y_LAGS,
            rain_lags=RAIN_LAGS,
            extra_cols=EXTRA_COLS_ACTIVE,
            joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
            extra_lags=EXTRA_LAGS,
            static_feat=static_feat,
            max_rows=MAX_TRAIN_ROWS,
            seed=seed_base,
        )
        x_val, y_val, y_val_aux, rows_pos_val, rows_kept_val = build_samples(
            va_events,
            start_t=START_T,
            y_lags=Y_LAGS,
            rain_lags=RAIN_LAGS,
            extra_cols=EXTRA_COLS_ACTIVE,
            joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
            extra_lags=EXTRA_LAGS,
            static_feat=static_feat,
            max_rows=MAX_VAL_ROWS,
            seed=seed_base + 1,
        )
        rollout_train_data, rollout_pos_train, rollout_kept_train = build_rollout_samples(
            tr_events,
            start_t=START_T,
            rollout_steps=ROLLOUT_STEPS,
            y_lags=Y_LAGS,
            rain_lags=RAIN_LAGS,
            extra_cols=EXTRA_COLS_ACTIVE,
            joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
            extra_lags=EXTRA_LAGS,
            max_samples=ROLLOUT_MAX_SAMPLES,
            seed=seed_base + 11,
        )
        log(
            f"fold {fold_idx+1}: train_samples={rows_kept_train}/{rows_pos_train} "
            f"val_samples={rows_kept_val}/{rows_pos_val} feat_dim={x_train.shape[1]} "
            f"rollout_samples={rollout_kept_train}/{rollout_pos_train}"
        )
        scaler_mean, scaler_std = fit_scaler(x_train)
        x_train = apply_scaler(x_train, scaler_mean, scaler_std)
        x_val = apply_scaler(x_val, scaler_mean, scaler_std)
        epoch_eval_events = va_events
        if EPOCH_OOF_MAX_EVENTS > 0:
            epoch_eval_events = va_events[:EPOCH_OOF_MAX_EVENTS]
        if OOF_LB_EVAL and EPOCH_OOF_EVAL_EVERY > 0 and len(epoch_eval_events) > 0:
            def _epoch_oof_eval_fn(model_cur):
                score, _ = evaluate_oof_lb_equivalent(
                    events=epoch_eval_events,
                    model=model_cur,
                    scaler_mean=scaler_mean,
                    scaler_std=scaler_std,
                    target=TARGET,
                    start_t=START_T,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=EXTRA_COLS_ACTIVE,
                    joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
                    extra_lags=EXTRA_LAGS,
                    static_feat=static_feat,
                )
                return float(score)
        else:
            _epoch_oof_eval_fn = None
        fit_result = train_one_fold(
            x_train,
            y_train,
            y_train_aux,
            x_val,
            y_val,
            y_val_aux,
            fold_idx,
            n_nodes=len(node_ids),
            edge_index_np=edge_index_np,
            y_lags=Y_LAGS,
            rain_lags=RAIN_LAGS,
            extra_cols=EXTRA_COLS_ACTIVE,
            joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
            extra_lags=EXTRA_LAGS,
            epoch_oof_eval_every=EPOCH_OOF_EVAL_EVERY,
            epoch_oof_eval_fn=_epoch_oof_eval_fn,
            aux_loss_weight=AUX_LOSS_WEIGHT,
            rollout_train_data=rollout_train_data,
            scaler_mean=scaler_mean,
            scaler_std=scaler_std,
            static_feat=static_feat,
        )
        model = fit_result["model"]
        best_rmse = float(fit_result["best_val_rmse"])
        sample_val_scores.append(best_rmse)
        selected_checkpoint = "val"
        if OOF_LB_EVAL:
            val_checkpoint_oof, _ = evaluate_oof_lb_equivalent(
                events=va_events,
                model=model,
                scaler_mean=scaler_mean,
                scaler_std=scaler_std,
                target=TARGET,
                start_t=START_T,
                y_lags=Y_LAGS,
                rain_lags=RAIN_LAGS,
                extra_cols=EXTRA_COLS_ACTIVE,
                joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
                extra_lags=EXTRA_LAGS,
                static_feat=static_feat,
            )
            oof_lb_score = float(val_checkpoint_oof)
            epoch_checkpoint_oof = float("nan")
            selected_state = fit_result["best_val_state"]
            if fit_result["best_oof_state"] is not None:
                model.load_state_dict(fit_result["best_oof_state"])
                epoch_checkpoint_oof, _ = evaluate_oof_lb_equivalent(
                    events=va_events,
                    model=model,
                    scaler_mean=scaler_mean,
                    scaler_std=scaler_std,
                    target=TARGET,
                    start_t=START_T,
                    y_lags=Y_LAGS,
                    rain_lags=RAIN_LAGS,
                    extra_cols=EXTRA_COLS_ACTIVE,
                    joint_aux_cols=JOINT_AUX_COLS_ACTIVE,
                    extra_lags=EXTRA_LAGS,
                    static_feat=static_feat,
                )
                if np.isfinite(epoch_checkpoint_oof) and (not np.isfinite(oof_lb_score) or epoch_checkpoint_oof < oof_lb_score):
                    selected_checkpoint = "epoch_oof"
                    selected_state = fit_result["best_oof_state"]
                    oof_lb_score = float(epoch_checkpoint_oof)
            model.load_state_dict(selected_state)
            tracked_epoch_oof_std = fit_result["best_epoch_oof_std"] if np.isfinite(fit_result["best_epoch_oof_std"]) else float("nan")
            log(
                f"fold {fold_idx+1} checkpoint_select: val_best_std={val_checkpoint_oof:.6f} "
                f"epoch_oof_best_std={epoch_checkpoint_oof:.6f} "
                f"epoch_oof_tracked={tracked_epoch_oof_std:.6f} chosen={selected_checkpoint}"
            )
        else:
            oof_lb_score = float("nan")
        oof_lb_scores.append(oof_lb_score)
        log(
            f"fold {fold_idx+1} summary: sample_std={best_rmse/float(STD_DEV[TARGET]):.6f} "
            f"oof_lb_std={oof_lb_score:.6f}"
        )
        fold_states.append(
            {
                "fold_idx": fold_idx,
                "val_ids": list(val_ids),
                "model": model,
                "scaler_mean": scaler_mean,
                "scaler_std": scaler_std,
                "node_means": means,
                "extra_means": extra_means,
                "static_cols": list(static_cols_active),
                "static_dim": int(static_feat.shape[1]),
                "selected_checkpoint": selected_checkpoint,
            }
        )
        del x_train, y_train, y_train_aux, x_val, y_val, y_val_aux, rollout_train_data, tr_events, va_events
        gc.collect()
    mean_sample_rmse = float(np.mean(sample_val_scores))
    mean_sample_std = mean_sample_rmse / float(STD_DEV[TARGET])
    valid_oof = [x for x in oof_lb_scores if np.isfinite(x)]
    mean_oof_lb = float(np.mean(valid_oof)) if valid_oof else float("nan")
    log(
        f"CV done: sample_rmse={mean_sample_rmse:.6f} sample_std={mean_sample_std:.6f} "
        f"oof_lb_std={mean_oof_lb:.6f} executed_folds={[i+1 for i in fold_indices]}"
    )
    log("Step 3: recursive prediction (fold average)")
    if not fold_states:
        log("no trained folds; writing fallback submission")
        build_submission(
            sample_path=sample_path,
            pred_df=pd.DataFrame(columns=KEY_COLS + ["water_level"]),
            output_path=OUTPUT_PATH,
        )
        log(f"submission saved: {OUTPUT_PATH}")
        return
    # Use average node means from folds for test fill
    avg_means = np.mean(np.stack([st["node_means"] for st in fold_states], axis=0), axis=0).astype(np.float32)
    avg_extra_means = {
        col: np.mean(
            np.stack([st["extra_means"][col] for st in fold_states], axis=0),
            axis=0,
        ).astype(np.float32)
        for col in EXTRA_COLS_ACTIVE
    }
    test_events = []
    for eid in test_ids:
        ev = test_events_all[eid]
        ev_use = {
            "event_id": ev["event_id"],
            "t": ev["t"],
            "y": ev["y"].copy(),
            "rain": ev["rain"],
            "extra": {k: v.copy() for k, v in ev["extra"].items()},
        }
        test_events.append(ev_use)
    fill_missing(test_events, avg_means, avg_extra_means)
    pred_df = build_prediction_df(
        model_id=model_id,
        node_type=node_type,
        events=test_events,
        node_ids=node_ids,
        fold_states=fold_states,
        static_feat=static_feat,
    )
    pred_df = pred_df.sort_values(KEY_COLS).reset_index(drop=True)
    log(f"prediction rows={len(pred_df)}")
    if SAVE_PRED_TABLE:
        pred_table_path = Path(PRED_TABLE_PATH)
        pred_table_path.parent.mkdir(parents=True, exist_ok=True)
        pred_df.to_csv(pred_table_path, index=False)
        log(f"prediction table saved: {pred_table_path}")
    build_submission(
        sample_path=sample_path,
        pred_df=pred_df,
        output_path=OUTPUT_PATH,
    )
    log(f"submission saved: {OUTPUT_PATH}")
    if SAVE_BUNDLE:
        log("Step 4: save bundle outputs")
        BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
        oof_df = build_oof_prediction_df(
            model_id=model_id,
            node_type=node_type,
            train_events_all=train_events_all,
            node_ids=node_ids,
            fold_states=fold_states,
            static_feat=static_feat,
        )
        test_df = pred_df[KEY_COLS + ["water_level"]].copy()
        test_df = test_df.rename(columns={"water_level": "water_level_pred"})
        # Add row_id when resolvable to maximize compatibility with ensemble merger.
        try:
            sample_df_for_rowid = pd.read_csv(sample_path, usecols=["row_id"] + KEY_COLS)
            test_df = test_df.merge(sample_df_for_rowid, on=KEY_COLS, how="left")
        except Exception:
            pass
        oof_path = save_table_auto(oof_df, BUNDLE_DIR / "oof_predictions")
        test_path = save_table_auto(test_df, BUNDLE_DIR / "test_predictions")
        submission_copy_path = BUNDLE_DIR / "submission.csv"
        pd.read_csv(OUTPUT_PATH).to_csv(submission_copy_path, index=False)
        cv_summary = pd.DataFrame(
            [
                {
                    "model_id": model_id,
                    "node_type": node_type,
                    "target_col": TARGET_COL,
                    "n_folds": N_FOLDS,
                    "executed_folds": ",".join(str(i + 1) for i in fold_indices),
                    "event_split_seed": EVENT_SPLIT_SEED,
                    "sample_rmse": mean_sample_rmse,
                    "sample_std": mean_sample_std,
                    "oof_lb_std": mean_oof_lb,
                    "oof_rows": len(oof_df),
                    "test_rows": len(test_df),
                }
            ]
        )
        cv_summary_path = BUNDLE_DIR / "cv_summary.csv"
        cv_summary.to_csv(cv_summary_path, index=False)
        manifest = {
            "schema_version": "ufb_ensemble_v1",
            "bundle_name": BUNDLE_NAME,
            "source_notebook": "physics-informed-gnn-model2-1d-static-graph-attention-multistep-rollout-rain-gate-train-predict",
            "model_family": "physics_informed_gnn",
            "model_id": model_id,
            "node_type": node_type,
            "target_col": TARGET_COL,
            "joint_aux_cols": list(JOINT_AUX_COLS_ACTIVE),
            "static_features_enable": bool(STATIC_FEATURES_ENABLE),
            "static_cols": list(static_cols_active),
            "static_dim": int(static_feat.shape[1]),
            "aux_loss_weight": float(AUX_LOSS_WEIGHT),
            "rollout_loss_enable": bool(ROLLOUT_LOSS_ENABLE),
            "rollout_steps": int(ROLLOUT_STEPS),
            "rollout_weight": float(ROLLOUT_WEIGHT),
            "n_folds": N_FOLDS,
            "executed_folds_count": len(fold_indices),
            "executed_folds": [int(i + 1) for i in fold_indices],
            "event_split_seed": EVENT_SPLIT_SEED,
            "oof_predictions_path": oof_path.name,
            "test_predictions_path": test_path.name,
            "submission_path": submission_copy_path.name,
            "cv_summary_path": cv_summary_path.name,
            "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        manifest_path = BUNDLE_DIR / "bundle_manifest.json"
        manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
        log(f"bundle saved: {BUNDLE_DIR}")
        log(f"bundle files: {oof_path.name}, {test_path.name}, {manifest_path.name}, {cv_summary_path.name}, {submission_copy_path.name}")
    log(f"done elapsed={time.time()-t0:.1f}s")


if __name__ == "__main__":
    main()


ストリーミング出力は最後の 5000 行に切り捨てられました。
[17:30:56] oof-lb eval progress: 11/14 event_std_rmse=0.165557
[17:31:03] oof-lb eval progress: 12/14 event_std_rmse=0.123530
[17:31:04] oof-lb eval progress: 13/14 event_std_rmse=0.049553
[17:31:07] oof-lb eval progress: 14/14 event_std_rmse=0.064051
[17:31:07] fold=2 epoch=80/300 oof_lb_std@epoch=0.083276
[17:31:49] fold=2 epoch=81/300 train_rmse=0.007866 val_rmse=0.015110 val_std=0.004734 lr=7.50e-05 rollout_mse=0.000176
[17:32:31] fold=2 epoch=82/300 train_rmse=0.008111 val_rmse=0.015022 val_std=0.004706 lr=7.50e-05 rollout_mse=0.000143
[17:33:12] fold=2 epoch=83/300 train_rmse=0.008548 val_rmse=0.015055 val_std=0.004717 lr=7.50e-05 rollout_mse=0.000177
[17:33:54] fold=2 epoch=84/300 train_rmse=0.008593 val_rmse=0.014963 val_std=0.004688 lr=7.50e-05 rollout_mse=0.000156
[17:34:36] fold=2 epoch=85/300 train_rmse=0.007801 val_rmse=0.015014 val_std=0.004704 lr=7.50e-05 rollout_mse=0.000142
[17:34:39] oof-lb eval progress: 1/14 event_std_rmse=0.117008
[1

In [5]:
# =========================
# Colab: finalize & persist (NON-rain3 version)
# Run AFTER the script finishes
# =========================
from pathlib import Path
import os, time, shutil, json

PROJECT_ROOT = Path("/content/drive/MyDrive/ufb")
STAMP = time.strftime("run_%Y%m%d_%H%M%S")
SAVE_DIR = PROJECT_ROOT / "runs" / STAMP
SAVE_DIR.mkdir(parents=True, exist_ok=True)

cwd = Path.cwd()

def abs_path(p: str) -> Path:
    pp = Path(p)
    return pp if pp.is_absolute() else (cwd / pp)

def safe_copy_file(src: Path, dst_dir: Path):
    if src.exists() and src.is_file():
        dst = dst_dir / src.name
        shutil.copy2(src, dst)
        return str(dst)
    return None

copied = {}

# 1) submission.csv
output_path = abs_path(os.environ.get("OUTPUT_PATH", "submission.csv"))
p = safe_copy_file(output_path, SAVE_DIR)
copied["submission_csv"] = p

# 2) pred table (NON-rain3 default)
pred_table_path = abs_path(os.environ.get("PRED_TABLE_PATH", "pred_physgnn_m2_1d_5fold_static_attn.csv"))
p = safe_copy_file(pred_table_path, SAVE_DIR)
copied["pred_table"] = p

# 3) bundle dir: prefer env, else infer from BUNDLE_NAME, else search for bundle_manifest.json
bundle_candidates = []

env_bd = os.environ.get("BUNDLE_DIR", "").strip()
if env_bd:
    bundle_candidates.append(abs_path(env_bd))

try:
    if "BUNDLE_NAME" in globals():
        bn = str(globals()["BUNDLE_NAME"])
        bundle_candidates += [
            abs_path(f"ensemble/artifacts/model_bundles/{bn}"),
            abs_path(f"model_bundles/{bn}"),
            abs_path(f"/content/model_bundles/{bn}"),
        ]
except Exception:
    pass

# common fixed fallback (non-rain3 default name used in your script)
bundle_candidates += [
    abs_path("ensemble/artifacts/model_bundles/physics_informed_gnn_m2_1d_5fold_static_attn"),
]

def looks_like_bundle(p: Path) -> bool:
    return p.is_dir() and (p / "bundle_manifest.json").exists()

bundle_dir = next((p for p in bundle_candidates if looks_like_bundle(p)), None)

# if still not found, search under cwd for bundle_manifest.json
if bundle_dir is None:
    hits = list(cwd.rglob("bundle_manifest.json"))[:50]
    if hits:
        bundle_dir = hits[0].parent

if bundle_dir is not None and looks_like_bundle(bundle_dir):
    dst_bundle = SAVE_DIR / "bundle"
    shutil.copytree(bundle_dir, dst_bundle, dirs_exist_ok=True)
    copied["bundle_dir"] = str(dst_bundle)
else:
    copied["bundle_dir"] = None

# 4) save manifest
manifest = {
    "saved_at": STAMP,
    "save_dir": str(SAVE_DIR),
    "cwd": str(cwd),
    "resolved": {
        "OUTPUT_PATH_env": os.environ.get("OUTPUT_PATH"),
        "PRED_TABLE_PATH_env": os.environ.get("PRED_TABLE_PATH"),
        "BUNDLE_DIR_env": os.environ.get("BUNDLE_DIR"),
        "bundle_dir_detected": str(bundle_dir) if bundle_dir else None,
    },
    "copied": copied,
}
(SAVE_DIR / "finalize_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2))

print("✅ Finalize complete.")
print("Saved to:", SAVE_DIR)
print("Copied:")
for k, v in copied.items():
    print(f"  - {k}: {v}")

if copied.get("submission_csv") is None:
    print("\n⚠️ submission.csv not found. If your script logged 'submission saved: <path>', set:")
    print('  os.environ["OUTPUT_PATH"] = "<that path>"  # then re-run this cell')

if copied.get("pred_table") is None:
    print("\n⚠️ pred table not found. If your script logged 'prediction table saved: <path>', set:")
    print('  os.environ["PRED_TABLE_PATH"] = "<that path>"  # then re-run this cell')

if copied.get("bundle_dir") is None:
    print("\n⚠️ bundle not copied (bundle_manifest.json not found). If your script logged 'bundle saved: <path>', set:")
    print('  os.environ["BUNDLE_DIR"] = "<that path>"  # then re-run this cell')

✅ Finalize complete.
Saved to: /content/drive/MyDrive/ufb/runs/run_20260315_091917
Copied:
  - submission_csv: /content/drive/MyDrive/ufb/runs/run_20260315_091917/submission.csv
  - pred_table: None
  - bundle_dir: /content/drive/MyDrive/ufb/runs/run_20260315_091917/bundle

⚠️ pred table not found. If your script logged 'prediction table saved: <path>', set:
  os.environ["PRED_TABLE_PATH"] = "<that path>"  # then re-run this cell
